
# Dissecting exploration

What the animal does while it looks for a new solution — first the **shape of the cursor** it
produces, then three **population measures** of how varied that behaviour is.
`mouse_0403L` only: it is the one animal with a remapping phase.

**Everything is rebuilt from the packets.** Nothing here reads the older `output/cursor_shape/`
or `output/mouse_0403L/exploration/` files — those came from earlier versions of this analysis
with different windows and a different cursor. The loader, the tone law and the other-axis
simulation come from `_bmi.py`, which is the code validated in notebook 02.

**Five things to know before reading any panel**

1. Every neural analysis is in **6 Hz bins** (5 frames at 30 Hz) — the rate used by every other
   neural notebook in this repo.
2. The cursor is `freqToOutput`, the tone the speaker actually played, drawn in the
   representation `CURSOR_REP` picks (`hz` by default; `oct`, `brain` and `theta` also work).
3. **After a hit the speaker is clamped.** `frequencyOverride` holds it at exactly 8777.2 Hz
   for 0.97 s, starting 0.07 s *before* the reward frame — so the last warped trial column and
   ~58 % of the post-trial flank are a constant, identical on every trial and every session.
   That is the flat top, and `REWARD_TONE = "hide"` blanks it. It is not a smoothing artefact:
   nothing in §2 is smoothed.
4. The **other-axis simulation** is what the silent decoder axis was reading. It exists on
   remapping sessions only — a stable session never ran a second axis, so there is no fitted
   model for one.
5. **A participation ratio is reported only when its condition holds at least 5 time bins per
   neuron.** With 468 neurons that is 2340 bins = 6.5 min. Several conditions fail that test;
   they are left blank, never imputed. This is the most important caveat in the notebook, and
   §3 says exactly which conditions fail and what is used instead.

## 1 · Setup

In [ ]:

%matplotlib inline
import sys, time, warnings
from pathlib import Path
from types import SimpleNamespace
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.cluster.hierarchy import dendrogram
from scipy import stats

NB_DIR = Path.cwd() if (Path.cwd() / "_style.py").exists() else Path.cwd() / "wrapup_notebooks"
sys.path.insert(0, str(NB_DIR))
import _style as st, _draw as dw, _bmi as bmi
import importlib; importlib.reload(st); importlib.reload(dw); importlib.reload(bmi)
REPO = st.repo_root()
sys.path.insert(0, str(REPO / "utilities"))
import neural_analysis_utils as na
import cursor_analysis_utils as ca

# ================= SETTINGS =================
THEME      = "dark"            # "light" | "dark"
FONT_SCALE = 1
DATA_ROOT  = Path(r"C:\Data\datasets\bmi_packets\bmi_packets")
MOUSE      = "mouse_0403L"     # the only animal with a remapping phase
SR         = 30.0              # imaging / logger frame rate
BIN        = 5                 # frames per bin -> 6 Hz, the rate every neural analysis uses
CURSOR_REP = "hz"              # "hz" | "oct" | "brain" | "theta"  -- see 1.1
MISS_SEC   = 20.0              # a miss times out at 20 s; that is its effective end
INT_MAX_MIN = 10.0             # the intuitive block is clipped to its first N minutes

# --- cursor-shape window ---
PRE_SEC, POST_SEC, N_TRIAL_BINS = 1.0, 1.5, 50
MAX_NAN_FRAC = 0.10            # reject a trial whose window is more NaN than this
MIN_TRIALS   = 3               # a condition needs this many trials to contribute a mean
REWARD_TONE  = "draw"          # the frames where the SPEAKER is clamped at the reward tone:
                               # "hide" blanks them, "draw" leaves them in (see 2)
MIN_COL_FRAC = 0.50            # draw a column only where this fraction of its trials still
                               # has a cursor -- only bites when REWARD_TONE == "hide"
N_SHOW_DAYS  = 7              # fig01/fig02: sessions of each phase to draw (None = all)

# --- clustering ---
NORMALIZE_MODE = "none"        # "none" | "zscore" | "minmax" | "center"
PCA_VAR_TARGET = 0.95
K_RANGE        = range(2, 11)
N_CLUSTERS     = None             # None -> the silhouette maximum
K_SENSITIVITY  = [2, 3, 4, 5]
ABUND_N_DAYS   = 10            # fig18: first N remapping sessions, intuitive vs new axis

# --- the exploration measures ---
MANIFOLD_K   = 40              # PCs that define "the manifold" (see 3)
N_WARP       = 50              # bins each trial is warped to before correlating
PR_MIN_RATIO = 5.0             # time bins per neuron below which a PR is NOT reported
PAIR_EXACT_MAX = 500           # up to this many trials every pair is used; above it, sampled
N_PAIRS_MAX  = 20000
SLIDE_WIN_MIN, SLIDE_STEP_MIN = 7.0, 2.0     # see the 5x note printed below
SLIDE_REMAP_RANKS = [0, 6, 12]               # remapping days drawn one by one in fig15
# fig14 and fig16 average several sessions into one line. BUNDLE = how many sessions per
# line, N_BUNDLES = how many lines. BUNDLE = 1 gives one line per session again.
STABLE_BUNDLE, STABLE_N_BUNDLES = 3, 3       # fig14, over the stable-control sessions
REMAP_BUNDLE,  REMAP_N_BUNDLES  = 3, 5       # fig16, over the remapping sessions
# fig19: every trial cut into this many equal parts of its own UNWARPED length
SEG_PARTS = 3                  # 3 -> early / middle / late
SEG_CAP   = 3000               # trial bins kept per session for the segment PR
SEG_BOOT  = 200                # bootstrap draws (resampling sessions) behind each box
SEED       = 0
RECOMPUTE  = False             # True re-runs every cached pass

# --- colour ---
C_INT    = "#7f3f98"           # the intuitive axis, everywhere
C_NEW    = "#f7941d"           # the rotated ("new") axis, everywhere
C_BAD    = "#e8453c"           # a MISS trial
DAY_CMAP = "cool"              # the across-day ramp
C_PR, C_CORR, C_LEN = "#00c2d1", "#e45ca8", "#a8d84a"   # one per exploration measure
# ============================================

SRB = SR / BIN
P = st.init(notebook="03_dissect_exploration", theme=THEME, scale=FONT_SCALE)
P["intuitive"] = P["stable"] = C_INT
P["new_axis"] = P["remap"] = C_NEW
P["incorrect"] = C_BAD
C_HIT, C_MISS = P["correct"], C_BAD      # trials are HIT and MISS, never correct/incorrect
CACHE = st.figdir() / "cache"; CACHE.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)

MEASURES = {
    "pr":   dict(c=C_PR,   label="participation ratio", short="PR"),
    "corr": dict(c=C_CORR, label="trial-pair correlation (in the manifold)", short="r"),
    "len":  dict(c=C_LEN,  label="trajectory length (in the manifold)", short="length"),
}
print(f"bin = {BIN} frames at {SR:g} Hz  ->  {SRB:g} Hz   (the rate used by every neural "
      f"analysis and PCA in this repo)")
print(f"measure colours: PR {C_PR}   correlation {C_CORR}   length {C_LEN}   "
      f"| axes: intuitive {C_INT}, new {C_NEW}, miss {C_BAD}")

### 1.1 · The data layer

Sessions, trials, blocks and the cursor. A remapping session is `intuitive → new → washout`;
the washout is excluded from every comparison below, and the intuitive block is clipped to its
first `INT_MAX_MIN` minutes so that it is one regime rather than "everything before the
rotation".

In [ ]:

# The loader, the tone law and the other-axis simulation all come from `_bmi.py`, which holds
# the SAME code that was validated in notebook 02; pointing `sim_cache` at notebook 02's cache
# folder reuses the readouts it already fitted instead of refitting 15 sessions.
# Nothing here reads the older `output/cursor_shape/` or `output/.../exploration/` files:
# those were written by earlier versions of this analysis, with different windows and a
# different cursor, and mixing them would be silently wrong. Everything is rebuilt.
SIM_CACHE = REPO / "output" / "wrapup_notebooks" / "02_learning_and_remapping" / "cache"
bmi.init(REPO, DATA_ROOT, [MOUSE], rep=CURSOR_REP, sr=SR, bin=BIN,
         sim_cache=SIM_CACHE, day0_override={"mouse_0403L": None})
REP = bmi.rep()
TAB = bmi.state().TAB[MOUSE]
STABLE = bmi.stable_indices(MOUSE)        # sessions with trials, before the rotation phase
REMAP = bmi.remap_indices(MOUSE)
rel = lambda i: bmi.rel_day(MOUSE, int(i))


def trial_table(day_i):
    """One row per true BMI trial: its block, its effective end, and whether it is comparable.

    A hit ends at the reward (threshold crossing + ~0.4 s); a miss times out at MISS_SEC.
    `use` is False for intuitive-block trials past the first INT_MAX_MIN minutes, so that
    "the intuitive axis" is one regime and not "everything before the rotation"."""
    S = bmi.session(MOUSE, day_i)
    tr = S.trials
    eff = na.effective_trial_end(tr, reward_frames=S.rew, miss_dur_sec=MISS_SEC, sr=SR)
    facs = {float(b.factor) for b in S.blocks}
    is_remap = len(facs) > 1
    names = (["intuitive", "new", "washout"] if (is_remap and len(S.blocks) == 3)
             else ["single"] * len(S.blocks))
    blk = np.array(["none"] * len(tr), dtype=object)
    for nm, b in zip(names, S.blocks):
        blk[b.first_trial:b.last_trial + 1] = nm
    t = pd.DataFrame(dict(
        i=int(day_i), day=S.day, rel=rel(day_i), is_remap=is_remap,
        trial=np.arange(len(tr)), start=tr.start, end=tr.end, eff=eff,
        success=tr.success, factor=tr.factor, block=blk,
        outcome=np.where(tr.success == 1, "hit", np.where(tr.success == 0, "miss", "other")),
        dur=(eff - tr.start) / SR, t_min=tr.start / SR / 60.0))
    b0 = float(S.blocks[0].start_frame) / SR / 60.0
    t["use"] = ~((t.block == "intuitive") & (t.t_min > b0 + INT_MAX_MIN))
    return t


def block_frames(day_i, which):
    """[start, end) frames of one named block.

    Two of the names are clipped on purpose. `intuitive` is its first INT_MAX_MIN minutes, and
    `new_early` is the first INT_MAX_MIN minutes of the new-axis block -- the span-matched
    partner for it, because the new block runs for about an hour and a dimensionality
    estimated over an hour picks up slow drift that a ten-minute one cannot."""
    S = bmi.session(MOUSE, day_i)
    if len({float(b.factor) for b in S.blocks}) == 1:
        return (float(S.blocks[0].start_frame), bmi.block_end(S, len(S.blocks) - 1))
    k = {"intuitive": 0, "new": 1, "new_early": 1, "washout": 2}[which]
    s, e = float(S.blocks[k].start_frame), bmi.block_end(S, k)
    if which in ("intuitive", "new_early"):
        e = min(e, s + INT_MAX_MIN * 60 * SR)
    return (s, e)


def swarm_off(y, width, nbins=30):
    """Deterministic bee-swarm offsets: points sharing a y-bin are laid out side by side.

    Jitter hides density and overplots; a swarm shows it, and it is identical on every run."""
    y = np.asarray(y, float)
    off = np.zeros(len(y))
    g = np.isfinite(y)
    if g.sum() < 2:
        return off
    edges = np.linspace(np.nanmin(y[g]), np.nanmax(y[g]) + 1e-9, nbins + 1)
    b = np.clip(np.digitize(y, edges) - 1, 0, nbins - 1)
    cnt = np.bincount(b[g], minlength=nbins)
    step = 2.0 * width / max(cnt.max() - 1, 1)
    for k in np.flatnonzero(cnt):
        j = np.flatnonzero(g & (b == k))
        j = j[np.argsort(y[j])]
        off[j] = (np.arange(len(j)) - (len(j) - 1) / 2.0) * step
    return off


def dist_box(ax, x, v, color, *, logy=False, width=0.36, label_n=False):
    """A violin with its median and IQR. `swarm_off` is right for a few hundred points; at
    several thousand it degenerates into a barcode of quantised frame durations, and the
    density is the thing worth seeing."""
    v = np.asarray(v, float)
    v = v[np.isfinite(v) & ((v > 0) if logy else np.ones(len(v), bool))]
    if len(v) < 5:
        return 0
    z = np.log10(v) if logy else v
    pc = ax.violinplot([z], positions=[x], widths=width * 1.9, showextrema=False,
                       showmedians=False)
    for b in pc["bodies"]:
        b.set(facecolor=color, alpha=0.45, edgecolor=color, lw=1.2, zorder=3)
    q = np.percentile(z, [25, 50, 75])
    ax.plot([x, x], q[[0, 2]], color=P["fg"], lw=1.6, zorder=5)
    ax.plot([x - width * 0.6, x + width * 0.6], [q[1]] * 2, color=P["fg"], lw=2.6, zorder=6)
    return len(v)


def log_yticks(ax, ticks=(0.5, 1, 2, 5, 10, 20)):
    """Label a log10-transformed axis in the original units (violins need the transform)."""
    ax.set_yticks(np.log10(ticks))
    ax.set_yticklabels([f"{t:g}" for t in ticks])


def shade(c, f):
    """Blend a colour toward the foreground: a second series in the same measure family."""
    a = np.array(mpl.colors.to_rgb(c))
    b = np.array(mpl.colors.to_rgb(P["fg"]))
    return mpl.colors.to_hex(a * (1 - f) + b * f)


TR = pd.concat([trial_table(int(i)) for i in TAB.loc[TAB["n"] > 0, "i"]], ignore_index=True)
N_NEURONS = int(na.load_neural(bmi.state().R[MOUSE],
                               bmi.state().DAYS[MOUSE][int(STABLE[0])]).shape[0])
print(f"{MOUSE}: {len(TAB)} sessions | {len(STABLE)} stable control, {len(REMAP)} remapping | "
      f"{N_NEURONS} neurons on every day (one tracked ROI set) | cursor = {REP['label']}")
print(f"{len(TR)} true BMI trials over {TR.i.nunique()} sessions "
      f"(session {int(TR.i.min())} has none and is dropped)")
display(TR[TR.use].groupby(["is_remap", "block"]).outcome.value_counts().unstack(fill_value=0))


## 2 · Cursor shape

The window is `PRE_SEC` of real time, the trial warped to `N_TRIAL_BINS` columns, then
`POST_SEC` of real time. Only the trial is warped, so the baseline before it and the tone
after it keep their own clock. Nothing in this section is smoothed.

In [ ]:

# The warp is written out here rather than taken from `ca.extract_trial_windows`, because two
# things about that version are wrong for this figure and both were measured, not guessed:
#
#  1. IT POINT-SAMPLES. 74 % of trials last longer than N_TRIAL_BINS frames, so warping them
#     down is decimation, and picking one frame per column aliases 30 Hz noise straight into
#     the shape. Averaging the frames that fall inside each column does not. On this animal
#     the two differ by 0.19 sd of the trace on an average trial. This is also the answer to
#     "should the trace be smoothed first?": the aliasing is what smoothing would have been
#     patching over, and integrating the column removes it with no free parameter.
#  2. ITS COLUMNS ARE CLOSED, THE AXIS IS HALF-OPEN. Column j held fraction j/(n-1) of the
#     trial but was drawn at j/n, so the trial's last sample sat one column left of the
#     trial-end line and the line itself landed on the first POST-trial frame. Here column j
#     covers [j/n, (j+1)/n) and is drawn at j/n, so the whole window -- both flanks and the
#     trial -- is one uniform grid of 1/n steps and x = 1 is exactly the trial end.
SPEC = ca.WindowSpec(pre_sec=PRE_SEC, post_sec=POST_SEC, n_trial_bins=N_TRIAL_BINS, sr=SR)
SHAPE_TAG = (f"shape_{REP['name']}_p{PRE_SEC:g}s{POST_SEC:g}_b{N_TRIAL_BINS}"
             f"_n{MAX_NAN_FRAC:g}_v2")


def warp_trial(y, n):
    """The trial portion on n half-open columns: column j is the MEAN of the frames falling
    in [j/n, (j+1)/n) of the trial. A trial shorter than n frames is interpolated up."""
    y = np.asarray(y, float)
    L = len(y)
    if L >= n:
        e = np.round(np.linspace(0, L, n + 1)).astype(int)
        return np.array([np.nanmean(y[a:b]) if b > a else y[min(a, L - 1)]
                         for a, b in zip(e[:-1], e[1:])])
    return np.interp((np.arange(n) + 0.5) / n, (np.arange(L) + 0.5) / L, y)


def window_axis(spec):
    """x of every column. The trial occupies [0, 1); each flank frame is one more 1/n step,
    so the grid is uniform and the two dashed lines sit exactly on the trial edges."""
    n = spec.n_trial_bins
    return np.concatenate([np.arange(-spec.pre_n, 0) / n, np.arange(n) / n,
                           1.0 + np.arange(spec.post_n) / n])


AX = window_axis(SPEC)


def fill_gaps(x):
    """Interpolate the NaNs inside one piece, holding the edges. Only short gaps ever reach
    this -- a window that is more than MAX_NAN_FRAC missing is thrown away instead -- and it
    has to happen before the warp, or a column whose frames are all NaN would poison the
    average. The reward clamp is masked on its own matrix afterwards, so blanking it never
    goes through here."""
    x = np.asarray(x, float)
    g = np.isfinite(x)
    if g.all() or not g.any():
        return x
    i = np.arange(len(x))
    return np.interp(i, i[g], x[g])


def trial_windows(y, starts, ends, spec, max_nan_frac=MAX_NAN_FRAC):
    """(M, kept, rejects) for one trace. A trial is dropped when its window runs off the
    recording, when the trial is under 2 frames, when the window is more NaN than
    max_nan_frac, or when a NaN survives the gap fill."""
    y = np.asarray(y, float)
    T, rows, kept, rej = len(y), [], [], []
    for i, (s, e) in enumerate(zip(starts, ends)):
        if not (np.isfinite(s) and np.isfinite(e) and e > s):
            rej.append("bad_times"); continue
        s_i, e_i = int(round(s)), int(round(e))
        lo, hi = s_i - spec.pre_n, e_i + spec.post_n
        if lo < 0 or hi > T:
            rej.append("window_off_recording"); continue
        if e_i - s_i < 2:
            rej.append("trial_too_short"); continue
        if np.mean(~np.isfinite(y[lo:hi])) > max_nan_frac:
            rej.append("too_many_nan"); continue
        row = np.concatenate([fill_gaps(y[lo:s_i]),
                              warp_trial(fill_gaps(y[s_i:e_i]), spec.n_trial_bins),
                              fill_gaps(y[e_i:hi])])
        if not np.isfinite(row).all():
            rej.append("nan_after_fill"); continue
        rows.append(row)
        kept.append(i)
    return (np.array(rows) if rows else np.empty((0, spec.n_cols))), np.array(kept, int), rej


def build_shape_dataset():
    """One row per kept trial, three matrices wide: the cursor the animal drove, the
    other-axis simulation, and the fraction of each column the SPEAKER was clamped for."""
    Mr_all, Ms_all, Mc_all, metas, rej = [], [], [], [], []
    for i in TAB.loc[TAB["n"] > 0, "i"]:
        S = bmi.session(MOUSE, int(i))
        t = trial_table(int(i))
        st_, en_ = t.start.to_numpy(), t.eff.to_numpy()
        Mr, kr, rj = trial_windows(S.cur_raw, st_, en_, SPEC)
        rej += [dict(i=int(i), reason=r) for r in rj]
        if len(kr) == 0:
            continue
        # the clamp mask, warped the same way, over exactly the trials that were kept
        Mc, _, _ = trial_windows(S.ovr.astype(float), st_[kr], en_[kr], SPEC, max_nan_frac=1.0)
        Ms = np.full((len(kr), SPEC.n_cols), np.nan)
        if bmi.has_other_axis(MOUSE, int(i)):
            o = bmi.sim_cursor(bmi.simulate_other_axis(MOUSE, int(i)), "other", smooth=False)
            Mo, ko, _ = trial_windows(o, st_, en_, SPEC)
            pos = {int(k): j for j, k in enumerate(kr)}
            for j, k in enumerate(ko):
                if int(k) in pos:
                    Ms[pos[int(k)]] = Mo[j]
        meta = t.iloc[kr].reset_index(drop=True)
        meta["has_sim"] = np.isfinite(Ms).all(axis=1)
        mt = ca.shape_metrics_table(Mr, SPEC, durations=meta.dur.to_numpy())
        metas.append(pd.concat([meta, mt.reset_index(drop=True)], axis=1))
        Mr_all.append(Mr); Ms_all.append(Ms); Mc_all.append(Mc)
    return (np.vstack(Mr_all), np.vstack(Ms_all), np.vstack(Mc_all),
            pd.concat(metas, ignore_index=True), pd.DataFrame(rej))


_fp, _fc = CACHE / f"{SHAPE_TAG}.npz", CACHE / f"{SHAPE_TAG}_meta.csv"
if _fp.exists() and _fc.exists() and not RECOMPUTE:
    _z = np.load(_fp)
    M_REAL, M_SIM, M_CLAMP = _z["real"], _z["sim"], _z["clamp"]
    SH = pd.read_csv(_fc)
    SH["day"] = SH["day"].astype(str)
    REJ = pd.read_csv(CACHE / f"{SHAPE_TAG}_rej.csv")
else:
    _t0 = time.time()
    M_REAL, M_SIM, M_CLAMP, SH, REJ = build_shape_dataset()
    np.savez_compressed(_fp, real=M_REAL, sim=M_SIM, clamp=M_CLAMP)
    SH.to_csv(_fc, index=False); REJ.to_csv(CACHE / f"{SHAPE_TAG}_rej.csv", index=False)
    print(f"built in {time.time() - _t0:.0f}s")

# ---- the reward clamp -------------------------------------------------------------------
# `frequencyOverride` holds the speaker at exactly 8777.2 Hz for ~0.97 s, and it starts ~0.07 s
# BEFORE the reward frame we call the trial end. So the last warped trial column and most of
# the post-trial flank of a hit are one constant, identical on every trial of every session --
# that, and not any smoothing, is why the hit traces used to flatten onto the same value.
CLAMPED = M_CLAMP > 0.5
M_SHOW = np.where(CLAMPED, np.nan, M_REAL) if REWARD_TONE == "hide" else M_REAL
_hit = (SH.outcome == "hit").to_numpy()
CLAMP_FRAC = CLAMPED[_hit].mean(axis=0)              # per column, over hits
_first = int(np.argmax(CLAMP_FRAC > 0.5)) if (CLAMP_FRAC > 0.5).any() else len(AX)


def col_mean(A, min_frac=MIN_COL_FRAC):
    """Column mean + SEM of a set of windows, blanked where too few trials contribute.

    With the clamp hidden a hit has no cursor for ~1 s after the threshold crossing. Averaging
    whatever is left would draw a line made of a different, shrinking subset of trials at
    every column, so a column is reported only when at least `min_frac` of them are there."""
    A = np.asarray(A, float)
    n = np.isfinite(A).sum(0)
    keep = n >= max(min_frac * len(A), MIN_TRIALS)
    mu = np.where(keep, np.nanmean(A, axis=0), np.nan)
    sd = np.where(keep, np.nanstd(A, axis=0, ddof=1), np.nan)
    return mu, np.where(keep, sd / np.sqrt(np.maximum(n, 1)), np.nan), n


assert len(M_REAL) == len(SH) == len(M_SIM) == len(M_CLAMP), "matrices and metadata disagree"
print(f"window: {SPEC.pre_n} pre columns ({PRE_SEC:g} s) + {N_TRIAL_BINS} warped trial columns"
      f" + {SPEC.post_n} post columns ({POST_SEC:g} s) = {SPEC.n_cols}")
print(f"{len(SH)} trials kept of {len(TR)}  |  other-axis simulation available on "
      f"{int(SH.has_sim.sum())} of them (remapping sessions only)")
if len(REJ):
    print("rejected:", ", ".join(f"{r} x{int(n)}" for r, n in
                                 REJ.groupby('reason').size().items()))
print(f"\nREWARD CLAMP  on a hit the speaker is held at the reward tone for at least "
      f"{CLAMPED[_hit][:, SPEC.pre_n + N_TRIAL_BINS:].sum(1).mean() / SR:.2f} s of the "
      f"{POST_SEC:g} s post flank\n   (the flanks are the only part of the window in real "
      f"time, so they are the only part that can be read in seconds).\n   It covers "
      f"{CLAMPED[_hit][:, SPEC.trial_slice][:, -1].mean():.0%} of the LAST warped trial column "
      f"and {CLAMP_FRAC[SPEC.pre_n + N_TRIAL_BINS:].mean():.0%} of the post-trial flank.\n   "
      f"REWARD_TONE = {REWARD_TONE!r}: " + ("those frames are blanked, so a hit trace simply "
      f"stops at the reward\n   (x = {AX[_first]:.2f}) and the post flank below belongs to "
      f"misses, which earn no reward and are never clamped."
      if REWARD_TONE == "hide" else "those frames are drawn, so every hit trace flattens onto "
      f"{bmi.reward_level_value():.4g} after the trial."))
print("\nNOTE  a stable-control session only ever ran ONE decoder axis, so there is no second "
      "axis\n      that was ever live and no validated model for one: the other-axis "
      "simulation exists\n      on remapping sessions only. Every simulated panel below is "
      "therefore remapping-only.\n      The simulation is a model output and carries no "
      "speaker clamp, so it is never blanked.")


### 2.1 · The average cursor of each session

First `N_SHOW_DAYS` sessions of each phase — beyond ten the lines pile up and nothing is
readable. The y limits come from the traces that were actually drawn.

In [ ]:

FIGNAME = "fig01_mean_cursor_per_day"
ROWS = [("stable control", STABLE, "single"),
        ("remapping · new axis", REMAP, "new")]
pick_days = lambda idxs: list(idxs) if N_SHOW_DAYS is None else list(idxs)[:int(N_SHOW_DAYS)]


def shape_xticks(ax):
    """The flanks are in seconds and the trial is in normalised time: two different clocks on
    one axis, so the labels say which is which rather than pretending it is all one unit."""
    ax.set_xticks([-SPEC.pre_n / N_TRIAL_BINS, 0.0, 0.5, 1.0, SPEC.post_n / N_TRIAL_BINS + 1])
    ax.set_xticklabels([f"−{PRE_SEC:g} s", "start", "mid", "end", f"+{POST_SEC:g} s"])


def shape_frame(ax, show_y=True):
    """Trial start and end, the reward level, cursor zero. The y limits are NOT set here --
    they are set once at the end from the data that actually got drawn."""
    for x in (0.0, 1.0):
        ax.axvline(x, color=P["muted"], ls="--", lw=1.0, zorder=1)
    rw = bmi.reward_level_value()
    if np.isfinite(rw):
        ax.axhline(rw, color=P["correct"], lw=1.2, zorder=1)
    ax.axhline(bmi.base_level_value(), color=P["muted"], lw=0.9, ls=":", zorder=1)
    ax.set_yscale(REP["scale"])
    ax.set_ylabel(REP["label"] if show_y else "", fontsize=13 * FONT_SCALE)
    if not show_y:
        ax.tick_params(labelleft=False)
    ax.set_xlim(AX[0], AX[-1])
    shape_xticks(ax)


def data_ylim(curves, pad=0.07):
    """Limits from the values that were plotted, not from the representation's full range."""
    v = np.concatenate([np.asarray(c, float).ravel() for c in curves]) if curves else np.array([])
    v = v[np.isfinite(v)]
    if len(v) < 2:
        return None
    lo, hi = float(v.min()), float(v.max())
    if REP["scale"] == "log":
        s = (np.log(hi) - np.log(lo)) * pad
        return float(np.exp(np.log(lo) - s)), float(np.exp(np.log(hi) + s))
    s = (hi - lo) * pad
    return lo - s, hi + s


def apply_ylim(axes, lim):
    """Set the limits and keep only the landmark ticks that fall inside them."""
    if lim is None:
        return
    for ax in np.atleast_1d(np.asarray(axes, object)).ravel():
        ax.set_ylim(*lim)
        if REP["ticks"]:
            keep = [t for t in REP["ticks"] if lim[0] <= t <= lim[1]]
            if len(keep) >= 2:
                ax.set_yticks(keep)
                ax.set_yticklabels([REP["ticklabels"][REP["ticks"].index(t)] for t in keep])
                ax.minorticks_off()


def mark_clamp(ax):
    """Shade the span where most hits are inside the reward clamp, and ONLY that span: the
    clamp lasts about a second, so it ends well before the window does and the trace comes
    back afterwards. Shading to the edge would have labelled real cursor as reward tone."""
    k = np.flatnonzero(CLAMP_FRAC > 0.5)
    if not len(k):
        return
    a, b = AX[k[0]], AX[k[-1]]
    ax.axvspan(a, b, color=P["muted"], alpha=0.13, lw=0, zorder=0)
    ax.text((a + b) / 2, 0.02, "reward tone", transform=ax.get_xaxis_transform(), ha="center",
            va="bottom", color=P["muted"], fontsize=9.5 * FONT_SCALE)


fig, axes = plt.subplots(2, 2, figsize=(13.4, 7.6), sharex=True, sharey=True,
                         constrained_layout=True)
DRAWN = []
for r, (title, idxs, blk) in enumerate(ROWS):
    show = pick_days(idxs)
    cols = st.day_colors(len(show), cmap=DAY_CMAP)
    for c, oc in enumerate(["hit", "miss"]):
        ax = axes[r, c]
        n_days = 0
        for col, i in zip(cols, show):
            m = ((SH.i == i) & (SH.outcome == oc) & (SH.block == blk)).to_numpy()
            if m.sum() < MIN_TRIALS:
                continue
            mu, _, _ = col_mean(M_SHOW[m])
            ax.plot(AX, mu, color=col, lw=1.6, alpha=0.9, zorder=3)
            DRAWN.append(mu); n_days += 1
        shape_frame(ax, show_y=(c == 0))
        if oc == "hit":
            mark_clamp(ax)
        ax.set_title(f"{title} · {oc}  ({n_days} sessions)", fontsize=13 * FONT_SCALE)
        if r == 1:
            ax.set_xlabel("trial, time-warped (flanks in real time)")
    sm = mpl.cm.ScalarMappable(cmap=st.day_cmap(cmap=DAY_CMAP),
                               norm=mpl.colors.Normalize(rel(show[0]), rel(show[-1])))
    cb = fig.colorbar(sm, ax=axes[r, :], fraction=0.030, pad=0.012)
    cb.set_label("session", fontsize=12 * FONT_SCALE)
    cb.outline.set_visible(False)
YLIM = data_ylim(DRAWN)
apply_ylim(axes, YLIM)
dw.label_panel(axes[0, 0], "a", color=P["fg"])
dw.label_panel(axes[0, 1], "b", color=P["fg"])
fig.suptitle(f"The cursor the animal drove, averaged over the trials of each session "
             f"(first {len(pick_days(STABLE))} of each phase)", fontsize=16 * FONT_SCALE)
st.save_fig(fig, FIGNAME); plt.show()

_h = SH[(SH.outcome == "hit") & (SH.block.isin(["single", "new"]))]
print(f"median hit duration: stable {_h[_h.block == 'single'].dur.median():.2f} s, "
      f"remapping new axis {_h[_h.block == 'new'].dur.median():.2f} s  "
      f"-- the trial portion is warped to {N_TRIAL_BINS} columns, so a change in SHAPE here "
      f"is not a change in duration; the flanks are the only part in real time.")
print(f"y limits {YLIM[0]:,.4g}–{YLIM[1]:,.4g} ({REP['label']}), taken from the "
      f"{len(DRAWN)} traces drawn;\nthe representation's own range "
      f"{REP['lim'][0]:,.4g}–{REP['lim'][1]:,.4g} is NOT used." if YLIM else
      "nothing was drawn, so there are no limits to report")

In [ ]:

FIGNAME = "fig02_mean_other_axis_simulation_per_day"
# During the new-axis block the axis that is NOT live is the intuitive one, so this is the
# old solution, read out while the animal was driving the new one. The mouse never heard it,
# and a model output has no speaker clamp, so nothing here is blanked.
SHOW_R = pick_days(REMAP)
fig, axes = plt.subplots(1, 2, figsize=(13.4, 4.2), sharex=True, sharey=True,
                         constrained_layout=True)
cols = st.day_colors(len(SHOW_R), cmap=DAY_CMAP)
DRAWN2 = []
for c, oc in enumerate(["hit", "miss"]):
    ax = axes[c]
    n_days = 0
    for col, i in zip(cols, SHOW_R):
        m = ((SH.i == i) & (SH.outcome == oc) & (SH.block == "new") & SH.has_sim).to_numpy()
        if m.sum() < MIN_TRIALS:
            continue
        mu, _, _ = col_mean(M_SIM[m])
        ax.plot(AX, mu, color=col, lw=1.6, alpha=0.9, zorder=3)
        DRAWN2.append(mu); n_days += 1
    shape_frame(ax, show_y=(c == 0))
    ax.set_title(f"other-axis simulation · {oc}  ({n_days} sessions)", fontsize=13 * FONT_SCALE)
    ax.set_xlabel("trial, time-warped (flanks in real time)")
apply_ylim(axes, data_ylim(DRAWN2))
sm = mpl.cm.ScalarMappable(cmap=st.day_cmap(cmap=DAY_CMAP),
                           norm=mpl.colors.Normalize(rel(SHOW_R[0]), rel(SHOW_R[-1])))
cb = fig.colorbar(sm, ax=axes, fraction=0.030, pad=0.012)
cb.set_label("session", fontsize=12 * FONT_SCALE); cb.outline.set_visible(False)
dw.label_panel(axes[0], "a", color=P["fg"]); dw.label_panel(axes[1], "b", color=P["fg"])
fig.suptitle("What the INTUITIVE axis was reading while the animal drove the new one "
             "(simulation — never heard)", fontsize=16 * FONT_SCALE)
st.save_fig(fig, FIGNAME); plt.show()

_v = [bmi.simulate_other_axis(MOUSE, int(i)).val for i in REMAP]
_v = [v for v in _v if v]
print(f"the simulation is a fitted readout, held out on the washout over {len(_v)} sessions: "
      f"r(M) = {np.min([v['r_M'] for v in _v]):.2f}–{np.max([v['r_M'] for v in _v]):.2f}, "
      f"r(tone) = {np.min([v['r_hz'] for v in _v]):.2f}–{np.max([v['r_hz'] for v in _v]):.2f}.")
print(f"The cubic gate D = M^3/||M||^2 amplifies the error upward, so frames at or above the "
      f"reward level are over-counted (simulated {np.median([v['frac_sim'] for v in _v]):.1%} "
      f"vs real {np.median([v['frac_real'] for v in _v]):.1%}): read these traces for WHERE "
      f"they sit, never for how many rewards the old axis would have earned.")

### 2.2 · Intuitive and new axis, on the same session

The two axes of one remapping session side by side, on hits only, for the sessions listed in
`CMP_RANKS` — spread across the remapping phase.

In [ ]:

FIGNAME = "fig03_intuitive_vs_new_hits"
CMP_RANKS = [0, 1, 4, 9]          # which remapping sessions appear, by rank
CMP_I = [int(REMAP[k]) for k in CMP_RANKS if k < len(REMAP)]


def mean_sem(M, mask):
    """Grand mean and SEM over the trials selected by `mask`, column by column."""
    A = M[np.asarray(mask)]
    n = int(np.isfinite(A).all(axis=1).sum())
    if len(A) < MIN_TRIALS:
        return None, None, 0
    mu, se, cnt = col_mean(A)
    return mu, se, int(cnt.max()) if len(cnt) else n


fig, axes = plt.subplots(1, len(CMP_I), figsize=(4.7 * len(CMP_I), 4.3), sharex=True,
                         sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes)
DRAWN3 = []
for ax, i, letter in zip(axes, CMP_I, "abcdef"):
    for blk, col, nm in [("intuitive", C_INT, "intuitive axis"), ("new", C_NEW, "new axis")]:
        m = ((SH.i == i) & (SH.outcome == "hit") & (SH.block == blk) & SH.use).to_numpy()
        mu, se, n = mean_sem(M_SHOW, m)
        if mu is None:
            continue
        ax.plot(AX, mu, color=col, lw=2.2, zorder=3, label=f"{nm} (n={n})")
        ax.fill_between(AX, mu - se, mu + se, color=col, alpha=0.28, lw=0, zorder=2)
        DRAWN3 += [mu - se, mu + se]
    shape_frame(ax, show_y=(ax is axes[0]))
    mark_clamp(ax)
    ax.set_title(f"session {rel(i)}   (remapping day {CMP_RANKS[list(CMP_I).index(i)] + 1})",
                 fontsize=13 * FONT_SCALE)
    ax.set_xlabel("trial, time-warped")
    dw.label_panel(ax, letter, color=P["fg"])
apply_ylim(axes, data_ylim(DRAWN3))
dw.legend_from(axes[0], [("intuitive axis", "line", C_INT, dict(lw=2.6)),
                         ("new axis", "line", C_NEW, dict(lw=2.6))],
               loc="upper left", fontsize=11 * FONT_SCALE)
fig.suptitle("Hits on a remapping session: the cursor the animal drove, before and after the "
             "rotation", fontsize=16 * FONT_SCALE)
st.save_fig(fig, FIGNAME); plt.show()
print(f"intuitive = block 0 only, clipped to its first {INT_MAX_MIN:g} min; the washout is "
      f"excluded throughout. Mean ± SEM over trials.")

In [ ]:

FIGNAME = "fig04_intuitive_vs_new_hits_simulated"
# The colour follows WHICH AXIS IS BEING READ, exactly as in notebook 02's movie -- so the
# colours swap relative to fig03: during intuitive-block hits the silent axis is the NEW one,
# and during new-block hits it is the INTUITIVE one.
# The legend sits in its own strip ABOVE the row (fig.legend, not ax.legend) so that no panel
# has to give up space for it and the four stay the same size, side by side.
fig, axes = plt.subplots(1, len(CMP_I), figsize=(4.7 * len(CMP_I), 4.5), sharex=True,
                         sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes)
DRAWN4 = []
for ax, i, letter in zip(axes, CMP_I, "abcdef"):
    for blk, col in [("intuitive", C_NEW), ("new", C_INT)]:
        m = ((SH.i == i) & (SH.outcome == "hit") & (SH.block == blk) & SH.use
             & SH.has_sim).to_numpy()
        mu, se, n = mean_sem(M_SIM, m)
        if mu is None:
            continue
        ax.plot(AX, mu, color=col, lw=2.2, ls=(0, (5, 1.6)), zorder=3)
        ax.fill_between(AX, mu - se, mu + se, color=col, alpha=0.26, lw=0, zorder=2)
        DRAWN4 += [mu - se, mu + se]
    shape_frame(ax, show_y=(ax is axes[0]))
    ax.set_title(f"session {rel(i)}", fontsize=13 * FONT_SCALE)
    ax.set_xlabel("trial, time-warped")
    dw.label_panel(ax, letter, color=P["fg"])
apply_ylim(axes, data_ylim(DRAWN4))
fig.legend(handles=[Line2D([], [], color=C_NEW, lw=2.4, ls=(0, (5, 1.6)),
                           label="new axis, read during intuitive-block hits"),
                    Line2D([], [], color=C_INT, lw=2.4, ls=(0, (5, 1.6)),
                           label="intuitive axis, read during new-block hits")],
           loc="outside lower center", ncol=2, frameon=False, fontsize=11 * FONT_SCALE)
fig.suptitle("The same hits, read on the axis that was silent (other-axis simulation)",
             fontsize=16 * FONT_SCALE)
st.save_fig(fig, FIGNAME); plt.show()
print("Dashed = simulated. If the silent axis also rose on every hit, the two solutions are "
      "not separable;\nif it stayed flat, the animal was driving one axis and not the other.")


### 2.3 · Clustering the shapes

Ward on the PCA of the warped windows. The question the abundance figures answer is whether
the rotation changes *which* shapes the animal produces, not just their average.

In [ ]:

from scipy.cluster.hierarchy import fcluster
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.ndimage import gaussian_filter1d

# Cluster colours are their own family: they never share a panel with the axis or measure
# colours, so they are free to be a neutral qualitative set.
CLUST_COLORS = ["#56B4E9", "#ADADAD", "#CC79A7", "#F0E442", "#6a51a3", "#31a354",
                "#d95f0e", "#7fcdbb"]
METRICS_SHOW = ["pre_level", "peak", "range", "mean_level", "end_level", "net_change",
                "path_length_trial", "tortuosity", "roughness", "monotonicity",
                "n_peaks", "time_to_peak"]
SMOOTH_TEST_COLS = 2.0         # the sensitivity test below, in window columns


def run_clustering(outcome, k=None, smooth_cols=0.0):
    """Ward on the PCA of the warped windows -- one linkage, cut at every k.

    (`ca.silhouette_scan` rebuilds the linkage for each k; with ~8 000 trials that is nine
    full O(n^2) passes, so the cut is done here instead. Same clusters, a fraction of the
    time.)"""
    idx = SH.index[SH.outcome == outcome].to_numpy()
    X = ca.normalize_rows(M_REAL[idx], NORMALIZE_MODE)
    if smooth_cols > 0:
        X = gaussian_filter1d(X, smooth_cols, axis=1, mode="nearest")
    pr = ca.pca_reduce(X, PCA_VAR_TARGET)
    Z = ca.linkage_matrix(pr.scores)
    rows = []
    for kk in K_RANGE:
        if kk < 2 or kk >= len(idx):
            continue
        lab = fcluster(Z, t=kk, criterion="maxclust")
        rows.append(dict(k=kk, silhouette=float(silhouette_score(
            pr.scores, lab, sample_size=min(4000, len(idx)), random_state=SEED))))
    sil = pd.DataFrame(rows)
    kk = int(k or sil.loc[sil.silhouette.idxmax(), "k"])
    lab = fcluster(Z, t=kk, criterion="maxclust")
    sub = SH.loc[idx].reset_index(drop=True).copy()
    sub["cluster"] = lab
    return dict(outcome=outcome, idx=idx, sub=sub, X=X,
                Xshow=np.where(CLAMPED[idx], np.nan, X) if REWARD_TONE == "hide" else X,
                pca=pr, sil=sil, k=kk, labels=lab, linkage=Z)


CLU = {}
for _oc in ["hit", "miss"]:
    _t0 = time.time()
    CLU[_oc] = run_clustering(_oc, k=N_CLUSTERS)
    _c = CLU[_oc]
    print(f"{_oc:5s} n={len(_c['idx']):5d}  PCs for {PCA_VAR_TARGET:.0%} var = "
          f"{_c['pca'].n_kept:3d}  k={_c['k']}  sizes={np.bincount(_c['labels'])[1:]}  "
          f"({time.time() - _t0:.0f}s)")
print(f"\nclustering uses EVERY kept trial of {MOUSE} (stable, intuitive, new and washout) so "
      f"the clusters span\nthe whole repertoire; the blocks are separated afterwards, in the "
      f"abundance figures. Normalisation = {NORMALIZE_MODE!r}.")

# ---- SHOULD THE TRACE BE SMOOTHED FIRST? -------------------------------------------------
# No, and it already is not: `S.cur_raw` is the unsmoothed logger trace, and the clustering
# runs on the warped matrix straight out of it. The reason smoothing looks tempting is real
# though -- resampling a 30 Hz trace down to N_TRIAL_BINS columns is decimation, and a
# point-sampled decimation aliases. The fix for that is to INTEGRATE each column rather than
# to blur the result, which is what `warp_trial` does, and it costs no free parameter.
# Blurring afterwards is also the wrong operation geometrically: a column is 1/30 s in the
# flanks but trial_duration/N_TRIAL_BINS inside the trial, so one sigma spans two different
# amounts of time in the same row. Here is what it would do to the labels anyway:
for _oc in ["hit", "miss"]:
    _sm = run_clustering(_oc, k=CLU[_oc]["k"], smooth_cols=SMOOTH_TEST_COLS)
    print(f"{_oc:5s}: an extra {SMOOTH_TEST_COLS:g}-column Gaussian smooth moves the labels "
          f"by adjusted Rand = "
          f"{adjusted_rand_score(CLU[_oc]['labels'], _sm['labels']):.3f} "
          f"(1.0 = identical partition), sizes {np.bincount(_sm['labels'])[1:]}")

In [ ]:
FIGNAME = "fig05_cluster_diagnostics"
fig, axes = plt.subplots(2, 3, figsize=(14.2, 7.4), constrained_layout=True)
for row, oc in enumerate(["hit", "miss"]):
    c = CLU[oc]
    ax = axes[row, 0]
    ev = np.cumsum(c["pca"].explained_ratio)[:40]
    ax.plot(np.arange(1, len(ev) + 1), ev, "o-", ms=3.5, color=P["fg"])
    ax.axhline(PCA_VAR_TARGET, color=P["accent"], ls="--", lw=1.2)
    ax.axvline(c["pca"].n_kept, color=P["muted"], ls=":", lw=1.2)
    ax.set(xlabel="PC", ylabel="cumulative variance",
           title=f"{oc}: {c['pca'].n_kept} PCs reach {PCA_VAR_TARGET:.0%}")
    ax = axes[row, 1]
    ax.plot(c["sil"].k, c["sil"].silhouette, "o-", color=P["fg"], ms=5)
    ax.axvline(c["k"], color=P["accent"], ls="--", lw=1.4)
    ax.set(xlabel="k", ylabel="mean silhouette", title=f"{oc}: k = {c['k']}")
    ax = axes[row, 2]
    dendrogram(c["linkage"], truncate_mode="lastp", p=25, no_labels=True, ax=ax,
               color_threshold=(c["linkage"][-(c["k"] - 1), 2] if c["k"] > 1 else None),
               above_threshold_color=P["muted"])
    ax.set(title=f"{oc}: Ward dendrogram (last 25 merges)")
    ax.tick_params(labelbottom=False)
for a, letter in zip(axes[0], "abc"):
    dw.label_panel(a, letter, color=P["fg"])
fig.suptitle("Clustering the cursor shapes: how many clusters, and how well separated",
             fontsize=16 * FONT_SCALE)
st.save_fig(fig, FIGNAME); plt.show()
print(f"The silhouette maximum is at k = "
      f"{int(CLU['hit']['sil'].loc[CLU['hit']['sil'].silhouette.idxmax(), 'k'])} for hits, "
      f"but it almost always favours the coarsest split; N_CLUSTERS = {N_CLUSTERS} is used "
      f"throughout; fig07 shows what k = {K_SENSITIVITY} would give instead.")


In [ ]:

FIGNAME = "fig06_what_each_cluster_is"
fig, axes = plt.subplots(2, 3, figsize=(15.0, 8.0), constrained_layout=True)
_d6 = []
for row, oc in enumerate(["hit", "miss"]):
    c = CLU[oc]
    sub, lab = c["sub"], c["labels"]
    ks = np.unique(lab)
    # --- a: the mean window of each cluster ---
    ax = axes[row, 0]
    for j, kk in enumerate(ks):
        s = lab == kk
        m_, se, _ = col_mean(c["Xshow"][s])
        ax.plot(AX, m_, color=CLUST_COLORS[j], lw=2.0, zorder=3, label=f"C{kk} (n={s.sum()})")
        ax.fill_between(AX, m_ - se, m_ + se, color=CLUST_COLORS[j], alpha=0.25, lw=0)
        _d6 += [m_ - se, m_ + se]
    shape_frame(ax)
    if oc == "hit":
        mark_clamp(ax)
    ax.set(xlabel="trial, time-warped", title=f"{oc}: mean shape per cluster")
    ax.legend(fontsize=10 * FONT_SCALE, loc="upper left")
    # --- b: the median metric profile, z-scored ACROSS clusters ---
    ax = axes[row, 1]
    prof = sub.groupby("cluster")[METRICS_SHOW].median()
    z = ((prof - prof.mean()) / prof.std().replace(0, np.nan)).to_numpy()
    im = ax.imshow(z, cmap="coolwarm", vmin=-2, vmax=2, aspect="auto")
    ax.set_xticks(range(len(METRICS_SHOW)))
    ax.set_xticklabels(METRICS_SHOW, rotation=60, ha="right", fontsize=9 * FONT_SCALE)
    ax.set_yticks(range(len(prof))); ax.set_yticklabels([f"C{k}" for k in prof.index])
    ax.set(title=f"{oc}: median metric profile")
    cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cb.set_label("z across clusters", fontsize=10 * FONT_SCALE); cb.outline.set_visible(False)
    # --- c: duration, the thing the clusters mostly track ---
    ax = axes[row, 2]
    for j, kk in enumerate(ks):
        dist_box(ax, j, sub.dur[lab == kk].to_numpy(), CLUST_COLORS[j])
    ax.set_xticks(range(len(ks))); ax.set_xticklabels([f"C{k}" for k in ks])
    ax.set(ylabel="trial duration (s)",
           title=f"{oc}: duration by cluster (bar = median, line = IQR)")
apply_ylim([axes[0, 0], axes[1, 0]], data_ylim(_d6))
for a, letter in zip(axes[0], "abc"):
    dw.label_panel(a, letter, color=P["fg"])
fig.suptitle("What the clusters are", fontsize=16 * FONT_SCALE)
st.save_fig(fig, FIGNAME); plt.show()

# how much of the clustering is simply trial duration
for oc in ["hit", "miss"]:
    c = CLU[oc]
    d, l = c["sub"].dur.to_numpy(), c["labels"]
    if np.var(d) < 1e-9:
        print(f"{oc:5s}: every trial lasts {np.median(d):.1f} s — a miss ends at the timeout "
              f"by construction, so it has no duration for the clusters to track")
        continue
    grand = d.mean()
    ss_b = sum((d[l == k].mean() - grand) ** 2 * (l == k).sum() for k in np.unique(l))
    eta2 = ss_b / np.sum((d - grand) ** 2)
    med = ", ".join(f"C{k} {np.median(d[l == k]):.1f}s" for k in np.unique(l))
    print(f"{oc:5s}: duration explains eta^2 = {eta2:.2f} of the between-cluster variance "
          f"({med})")
print("A BLACK cell in panel b is a metric whose median is identical in every cluster: on "
      "hits that is\n`end_level`, because the last warped column of every hit is the same "
      "clamped reward tone.\n")
print("A high eta^2 means these are largely DURATION clusters: the trial portion is warped, "
      "but the\nflanks are in real time and the overall level still carries how long the "
      "animal took.")

In [ ]:

FIGNAME = "fig07_cluster_k_sensitivity"
# The chosen k is a decision, not a result: this is the same linkage cut at several k, so the
# hierarchy is visible and nothing below rests on one arbitrary number.
fig, axes = plt.subplots(2, len(K_SENSITIVITY), figsize=(3.6 * len(K_SENSITIVITY), 6.8),
                         sharex=True, sharey=True, constrained_layout=True)
_d7 = []
for row, oc in enumerate(["hit", "miss"]):
    c = CLU[oc]
    for col, kk in enumerate(K_SENSITIVITY):
        ax = axes[row, col]
        lab = fcluster(c["linkage"], t=kk, criterion="maxclust")
        for j, u in enumerate(np.unique(lab)):
            s = lab == u
            mu, _, _ = col_mean(c["Xshow"][s])
            ax.plot(AX, mu, color=CLUST_COLORS[j], lw=1.9, label=f"C{u} ({s.sum()})")
            _d7.append(mu)
        shape_frame(ax, show_y=(col == 0))
        ax.set_title(f"{oc} · k = {kk}" + ("  ←" if kk == c["k"] else ""),
                     fontsize=12 * FONT_SCALE)
        ax.legend(fontsize=8 * FONT_SCALE, loc="lower right")
        if row == 1:
            ax.set_xlabel("trial, time-warped")
apply_ylim(axes, data_ylim(_d7))
for a, letter in zip(axes[0], "abcdef"):
    dw.label_panel(a, letter, color=P["fg"])
fig.suptitle(f"The same linkage cut at several k (arrow = the k used everywhere else)",
             fontsize=16 * FONT_SCALE)
st.save_fig(fig, FIGNAME); plt.show()

In [ ]:
FIGNAME = "fig08_cluster_abundance_across_days"
# Which shapes the animal actually produces, session by session. The two remapping panels are
# the same sessions split by which decoder axis was live, so a change in repertoire at the
# rotation shows up as the two panels disagreeing.
PANELS = [("stable control", "single", STABLE),
          ("remapping · intuitive axis", "intuitive", REMAP),
          ("remapping · new axis", "new", REMAP)]
fig, axes = plt.subplots(2, 3, figsize=(15.0, 7.6), sharey=True, constrained_layout=True)
for row, oc in enumerate(["hit", "miss"]):
    sub = CLU[oc]["sub"]
    ks = np.unique(CLU[oc]["labels"])
    for col, (title, blk, idxs) in enumerate(PANELS):
        ax = axes[row, col]
        d = sub[(sub.block == blk) & sub.use & sub.i.isin(idxs)]
        ct = ca.cluster_composition(d, "cluster", "i", normalize="index")
        n_tr = d.groupby("i").size()
        ct = ct[n_tr.reindex(ct.index).fillna(0) >= MIN_TRIALS]
        x = [rel(i) for i in ct.index]
        for j, kk in enumerate(ks):
            y = ct[kk].to_numpy() if kk in ct.columns else np.zeros(len(ct))
            ax.plot(x, y, "o-", ms=4.5, lw=1.8, color=CLUST_COLORS[j], label=f"C{kk}")
        if blk != "single" and len(x):
            ax.set_xlim(min(x) - 0.6, max(x) + 0.6)
        ax.set_ylim(-0.03, 1.03)
        ax.set_title(f"{title} · {oc}  ({int(n_tr.sum())} trials)", fontsize=12.5 * FONT_SCALE)
        ax.grid(axis="y", alpha=0.35)
        if col == 0:
            ax.set_ylabel("fraction of that session's trials")
        if row == 1:
            ax.set_xlabel("session")
axes[0, 0].legend(fontsize=10 * FONT_SCALE, ncol=2, loc="upper left")
for a, letter in zip(axes[0], "abc"):
    dw.label_panel(a, letter, color=P["fg"])
fig.suptitle("How often each cursor shape occurs, across sessions", fontsize=16 * FONT_SCALE)
st.save_fig(fig, FIGNAME); plt.show()

_h = CLU["hit"]["sub"]
for blk, nm in [("intuitive", "intuitive"), ("new", "new axis")]:
    d = _h[(_h.block == blk) & _h.use]
    if len(d):
        f = d.cluster.value_counts(normalize=True).sort_index()
        print(f"remapping hits, {nm:9s}: " + "  ".join(f"C{k} {v:.0%}" for k, v in f.items()))
d = _h[_h.block == "single"]
print(f"stable hits{'':11s}: " + "  ".join(
    f"C{k} {v:.0%}" for k, v in d.cluster.value_counts(normalize=True).sort_index().items()))



### 2.4 · How long a trial lasts, and how the mix changes at the rotation

A miss has no duration of its own — it ends at the 20 s timeout by construction — so the
length distribution is a statement about hits.

In [ ]:

FIGNAME = "fig17_trial_length_distribution"
# Trial length is the thing the warping deliberately removes from every shape panel, so it
# gets its own figure: the clusters below turn out to be largely duration clusters, and the
# §3 trajectory length inherits the same confound.
LEN_BLOCKS = [("stable control", "single", STABLE, P["muted"]),
              ("intuitive axis", "intuitive", REMAP, C_INT),
              ("new axis", "new", REMAP, C_NEW)]
fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.5), constrained_layout=True)

# --- a: the distribution itself, as an ECDF (no bin width to argue about) ---
ax = axes[0]
for nm, blk, idxs, col in LEN_BLOCKS:
    d = SH[(SH.outcome == "hit") & (SH.block == blk) & SH.use & SH.i.isin(idxs)].dur.to_numpy()
    if len(d) < MIN_TRIALS:
        continue
    d = np.sort(d)
    ax.step(d, np.arange(1, len(d) + 1) / len(d), where="post", color=col, lw=2.2,
            label=f"{nm}  (n={len(d)}, median {np.median(d):.2f} s)")
ax.axvline(MISS_SEC, color=C_BAD, ls="--", lw=1.4)
ax.text(MISS_SEC, 0.04, " miss timeout", color=C_BAD, fontsize=9.5 * FONT_SCALE, ha="right",
        rotation=90, va="bottom")
ax.set_xscale("log")
ax.set_xticks([0.5, 1, 2, 5, 10, 20])
ax.set_xticklabels(["0.5", "1", "2", "5", "10", "20"])
ax.minorticks_off()
ax.set(xlabel="hit duration (s)", ylabel="fraction of hits at or below",
       title="by condition")
ax.legend(fontsize=9.5 * FONT_SCALE, loc="upper left")
ax.grid(alpha=0.3)

# --- b: the same split by cluster, which is what makes them duration clusters ---
ax = axes[1]
sub = CLU["hit"]["sub"]
ks = np.unique(CLU["hit"]["labels"])
for j, kk in enumerate(ks):
    dist_box(ax, j, sub.dur[sub.cluster == kk].to_numpy(), CLUST_COLORS[j], logy=True)
log_yticks(ax)
ax.set_xticks(range(len(ks)))
ax.set_xticklabels([f"C{k}\n(n={int((sub.cluster == k).sum())})" for k in ks])
ax.set(ylabel="hit duration (s)", title="by cluster (bar = median, line = IQR)")
ax.grid(axis="y", alpha=0.3)

# --- c: and across sessions, which is the part that could confound every day-wise panel ---
ax = axes[2]
for nm, blk, idxs, col in LEN_BLOCKS:
    d = SH[(SH.outcome == "hit") & (SH.block == blk) & SH.use & SH.i.isin(idxs)]
    if not len(d):
        continue
    g = d.groupby("i").dur
    keep = g.size() >= MIN_TRIALS
    x = [rel(i) for i in g.median().index[keep]]
    ax.plot(x, g.median()[keep], "o-", color=col, ms=5, lw=1.8, zorder=3, label=nm)
    ax.fill_between(x, g.quantile(0.25)[keep], g.quantile(0.75)[keep], color=col, alpha=0.18,
                    lw=0, zorder=2)
ax.set(xlabel="session", ylabel="hit duration (s)", title="median per session (band = IQR)")
ax.legend(fontsize=10 * FONT_SCALE, loc="best")
ax.grid(axis="y", alpha=0.3)
for a, letter in zip(axes, "abc"):
    dw.label_panel(a, letter, color=P["fg"])
fig.suptitle("How long a trial lasts  (hits; a miss ends at the 20 s timeout by "
             "construction)", fontsize=15 * FONT_SCALE)
st.save_fig(fig, FIGNAME); plt.show()

_m = SH[SH.outcome == "miss"].dur
print(f"misses: {len(_m)} trials, all {_m.min():.1f}–{_m.max():.1f} s — the timeout, so a "
      f"miss has no duration of its own and the miss clusters cannot be duration clusters.")
for nm, blk, idxs, _ in LEN_BLOCKS:
    d = SH[(SH.outcome == "hit") & (SH.block == blk) & SH.use & SH.i.isin(idxs)].dur
    if len(d):
        print(f"{nm:15s}: median {d.median():5.2f} s   IQR {d.quantile(.25):.2f}–"
              f"{d.quantile(.75):.2f}   p95 {d.quantile(.95):5.2f} s   n={len(d)}")

In [ ]:

FIGNAME = "fig18_cluster_mix_intuitive_vs_new"
# Every bar is one (session, axis) and sums to 1, so the bars say nothing about how many
# trials there were -- only about the MIX. The pair at the right pools all the sessions shown.
ABUND_I = list(REMAP)[:int(ABUND_N_DAYS)]
PAIR = [("intuitive", C_INT), ("new", C_NEW)]
W = 0.38
fig, axes = plt.subplots(2, 1, figsize=(13.6, 8.2), constrained_layout=True)
for row, oc in enumerate(["hit", "miss"]):
    ax = axes[row]
    sub = CLU[oc]["sub"]
    ks = np.unique(CLU[oc]["labels"])
    xs, lab_n = [], []
    for p, i in enumerate(ABUND_I + ["all"]):
        for q, (blk, acc) in enumerate(PAIR):
            d = sub[(sub.block == blk) & sub.use
                    & (sub.i.isin(ABUND_I) if i == "all" else (sub.i == i))]
            x = p + (q - 0.5) * W
            if len(d) < MIN_TRIALS:
                continue
            f = d.cluster.value_counts(normalize=True)
            bot = 0.0
            for j, kk in enumerate(ks):
                v = float(f.get(kk, 0.0))
                ax.bar(x, v, W * 0.92, bottom=bot, color=CLUST_COLORS[j],
                       edgecolor=acc, linewidth=1.6, zorder=3)
                bot += v
            ax.text(x, 0.012, blk[0].upper(), ha="center", va="bottom", color=P["bg"],
                    fontsize=10 * FONT_SCALE, fontweight="bold", zorder=6)
        xs.append(p)
        lab_n.append("all\nshown" if i == "all" else str(rel(int(i))))
    ax.set_xticks(xs); ax.set_xticklabels(lab_n)
    ax.set_ylim(0, 1.0); ax.set_xlim(-0.7, len(ABUND_I) + 0.7)
    ax.set_ylabel(f"fraction of {oc} trials")
    ax.set_title(f"{oc}: cluster mix, intuitive axis (I) vs new axis (N)",
                 fontsize=13 * FONT_SCALE)
    if row == 1:
        ax.set_xlabel("session")
    dw.label_panel(ax, "ab"[row], color=P["fg"])
axes[0].legend(handles=[mpl.patches.Patch(facecolor=CLUST_COLORS[j], label=f"C{k}")
                        for j, k in enumerate(np.unique(CLU["hit"]["labels"]))],
               fontsize=10 * FONT_SCALE, ncol=len(np.unique(CLU["hit"]["labels"])),
               loc="lower center", bbox_to_anchor=(0.5, 1.10), frameon=False)
fig.suptitle(f"Does the rotation change which shapes the animal produces? "
             f"(first {len(ABUND_I)} remapping sessions)", fontsize=16 * FONT_SCALE)
st.save_fig(fig, FIGNAME); plt.show()

for oc in ["hit", "miss"]:
    sub = CLU[oc]["sub"]
    row = []
    for blk, _ in PAIR:
        d = sub[(sub.block == blk) & sub.use & sub.i.isin(ABUND_I)]
        f = d.cluster.value_counts(normalize=True).sort_index()
        row.append(f"{blk:9s} " + " ".join(f"C{k} {v:.0%}" for k, v in f.items())
                   + f"  (n={len(d)})")
    print(f"{oc:5s}: " + "   |   ".join(row))
    a = sub[(sub.block == "intuitive") & sub.use & sub.i.isin(ABUND_I)]
    b = sub[(sub.block == "new") & sub.use & sub.i.isin(ABUND_I)]
    if len(a) >= 10 and len(b) >= 10:
        ct = pd.crosstab(pd.concat([pd.Series("intuitive", index=a.index),
                                    pd.Series("new", index=b.index)]),
                         pd.concat([a.cluster, b.cluster]))
        chi = stats.chi2_contingency(ct.to_numpy())
        print(f"       chi2 on the pooled 2 x {ct.shape[1]} table: p = {chi.pvalue:.3g} "
              f"(trials are not independent within a session — read this as a size, not a "
              f"test)")

## 3 · The exploration signature

Three measures, all in the same 6 Hz bins:

| | what it is | warped? | smoothed? |
|---|---|---|---|
| **participation ratio** | effective number of dimensions the population uses | — | no |
| **trial-pair correlation** | how alike two trials are, *inside the manifold* | yes | no |
| **trajectory length** | how far the state travels on a trial, inside the manifold | **no** | no |

The manifold is the top `MANIFOLD_K` PCs of the **whole session** — never of a window, which
would let the subspace follow the thing being measured. Correlation needs warping to compare
trials of different length, which is exactly what destroys length, so the third measure is
computed before any warping. Nothing is smoothed: smoothing injects shared autocorrelation,
which inflates both the correlation and the apparent dimensionality.

In [ ]:
# ---------------------------------------------------------------- measure 1: dimensionality
PR_BOOT = 8                    # bootstraps per matched PR
PR_NEURON_SUBSET = None        # int -> use this many neurons (the SAME ones everywhere), which
                               # lowers the 5x floor proportionally. None = all of them.
ITI_EXCLUDE_SEC = 10.0         # a baseline bin must be this far from any trial

NEU = (None if PR_NEURON_SUBSET is None
       else np.sort(np.random.default_rng(SEED).choice(N_NEURONS, int(PR_NEURON_SUBSET),
                                                       replace=False)))
N_USED = N_NEURONS if NEU is None else len(NEU)
MIN_BINS = int(np.ceil(PR_MIN_RATIO * N_USED))


def pr_of(X):
    """Participation ratio (sum l)^2 / sum l^2 of a (samples, neurons) matrix.

    PR is the "effective number of dimensions": 1 if all the variance is on one axis, N if it
    is spread evenly. It is a property of the covariance, so it is biased by how many samples
    estimated that covariance -- which is why nothing below compares two PRs computed from
    different numbers of bins."""
    Xc = X - X.mean(0)
    n = Xc.shape[0]
    lam = np.clip(np.linalg.eigvalsh((Xc.T @ Xc) / max(n - 1, 1)), 0.0, None)
    s = lam.sum()
    return float(s * s / np.sum(lam * lam)) if s > 0 else np.nan


def pr_matched(X, n_match, n_boot=PR_BOOT, seed=SEED):
    """PR at a FIXED sample count. Returns (mean, sd, valid).

    `valid` is False -- and the result NaN -- whenever there are fewer than n_match bins, or
    fewer than PR_MIN_RATIO bins per neuron. That second test is the hard one: a covariance
    estimated from fewer samples than ~5x its own dimension is dominated by sampling noise and
    its PR is not interpretable, so it is not reported at all."""
    n, p = X.shape
    if n_match is None:
        n_match = n
    if n < n_match or n_match < PR_MIN_RATIO * p:
        return np.nan, np.nan, False
    if n == n_match:
        return pr_of(X), 0.0, True
    rng = np.random.default_rng(seed)
    v = [pr_of(X[rng.choice(n, n_match, replace=False)]) for _ in range(n_boot)]
    return float(np.mean(v)), float(np.std(v, ddof=1)), True


# ------------------------------------------------- measure 2: how alike two trials are
def pair_corr(W, max_pairs=N_PAIRS_MAX, seed=SEED):
    """Correlation between every pair of warped trial trajectories, inside the manifold.

    `W` is (trials, N_WARP, k). Each trajectory is flattened to one (time x PC) vector,
    centred and scaled to unit norm, so the Pearson correlation of a pair is a dot product and
    ALL pairs are a single matrix product -- exact, and cheaper than sampling. Above
    PAIR_EXACT_MAX trials it falls back to `max_pairs` random pairs.

    Trials are warped (so trials of different length can be compared column by column) and NOT
    smoothed: smoothing would inflate the correlation by injecting shared autocorrelation."""
    W = np.asarray(W)
    good = np.isfinite(W).all(axis=(1, 2))
    W = W[good]
    n = len(W)
    if n < 3:
        return np.array([])
    Z = W.reshape(n, -1).astype(np.float64)
    Z -= Z.mean(1, keepdims=True)
    nrm = np.linalg.norm(Z, axis=1)
    nrm[nrm == 0] = 1.0
    Z /= nrm[:, None]
    if n <= PAIR_EXACT_MAX:
        return (Z @ Z.T)[np.triu_indices(n, 1)]
    rng = np.random.default_rng(seed)
    a, b = rng.integers(0, n, max_pairs), rng.integers(0, n, max_pairs)
    k = a != b
    return np.einsum("ij,ij->i", Z[a[k]], Z[b[k]])


# ------------------------------------------------- measure 3: how far the state travels
# `na.trajectory_length` -- the summed Euclidean step along the UNWARPED trajectory projected
# into the manifold. Warping is what makes measure 2 comparable, and it is also what destroys
# length, so this one is deliberately computed before any warping and without smoothing.
# It grows with trial duration; the printouts below give the duration correlation so that is
# never mistaken for a change in exploration.

def mean_ci(v, ci=0.95):
    """Mean and a normal 95 % interval on the mean (the shaded band in every measure panel)."""
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) < 2:
        return (np.nan, np.nan, np.nan, len(v))
    m = float(v.mean())
    h = 1.959963985 * float(v.std(ddof=1)) / np.sqrt(len(v))
    return (m, m - h, m + h, len(v))


print(f"PR uses {N_USED} of {N_NEURONS} neurons -> a PR needs at least {MIN_BINS} bins "
      f"({MIN_BINS / SRB / 60:.1f} min). Correlation and length have no such floor.")


In [ ]:
KINDS = ["session", "all_trials", "hit_trials", "iti"]
BLOCKS_OF = {int(i): (["intuitive", "new", "washout"] if bool(v) else ["single"])
             for i, v in zip(TAB.i, TAB.is_remap) if int(i) in set(TR.i)}
USED_BLOCKS = {"single", "intuitive", "new"}            # the washout is never compared
# `new_early` is a PR-only condition: the same ten minutes of task time as the intuitive
# block, so the two can be compared without the length of the block doing the work.
PR_BLOCKS_OF = {i: [b for b in bs if b in USED_BLOCKS] + (["new_early"] if "new" in bs else [])
                for i, bs in BLOCKS_OF.items()}


def day_bins(i):
    """Everything about one session that does NOT need the neural data: the bin masks."""
    S = bmi.session(MOUSE, i)
    t = trial_table(i)
    nb = S.nf // BIN
    inx = S.in_exp[:nb * BIN].reshape(nb, BIN).mean(1) > 0.5
    sb = np.round(t.start.to_numpy() / BIN).astype(int)
    eb = np.round(t.eff.to_numpy() / BIN).astype(int)
    ok = (sb >= 0) & (eb <= nb) & (eb - sb >= 3)
    m_iti = np.zeros(nb, bool)
    for s_, e_ in na.baseline_intervals(S.trials, S.nf, exclude_sec=ITI_EXCLUDE_SEC, sr=SR,
                                        reward_frames=S.rew, miss_dur_sec=MISS_SEC):
        m_iti[int(round(s_ / BIN)):int(round(e_ / BIN))] = True
    return SimpleNamespace(i=i, nb=nb, inx=inx, sb=sb, eb=eb, ok=ok, t=t, m_iti=m_iti & inx)


def block_bin_span(db, block):
    lo, hi = block_frames(db.i, block)
    m = np.zeros(db.nb, bool)
    m[max(int(round(lo / BIN)), 0):min(int(round(hi / BIN)), db.nb)] = True
    return m


def cond_mask(db, block, kind):
    """The bins that belong to one (block, kind) condition."""
    span = block_bin_span(db, block)
    if kind == "session":
        return span & db.inx
    if kind == "iti":
        return span & db.m_iti
    sel = db.ok & (db.t.block.to_numpy() == ("new" if block == "new_early" else block))
    sel &= db.t.use.to_numpy()
    if kind == "hit_trials":
        sel &= (db.t.outcome.to_numpy() == "hit")
    m = np.zeros(db.nb, bool)
    for s_, e_ in zip(db.sb[sel], db.eb[sel]):
        m[s_:e_] = True
    return m & span & db.inx


DB = {int(i): day_bins(int(i)) for i in sorted(set(TR.i))}

# ---- the match counts, decided BEFORE any neural data is touched ----
SIZE = pd.DataFrame([dict(i=i, block=b, kind=k, n=int(cond_mask(db, b, k).sum()))
                     for i, db in DB.items() for b in PR_BLOCKS_OF[i] for k in KINDS])
MATCH, DROPPED = {}, {}
for k in KINDS:
    s = SIZE[SIZE.kind == k]
    el = s[s.n >= MIN_BINS]
    MATCH[k] = int(el.n.min()) if len(el) else None
    DROPPED[k] = len(s) - len(el)
print(f"matched bin counts (every PR of a given kind is computed from exactly this many "
      f"randomly chosen bins, {PR_BOOT} times):")
for k in KINDS:
    s = SIZE[SIZE.kind == k]
    note = (f"match = {MATCH[k]} bins ({MATCH[k] / N_USED:.1f}x)"
            if MATCH[k] else "NO GROUP REACHES THE FLOOR — scenario dropped entirely")
    print(f"  {k:11s} sizes {int(s.n.min()):5d}–{int(s.n.max()):6d}   {note}"
          f"   | {DROPPED[k]}/{len(s)} day-blocks below {MIN_BINS} bins")
_bad = SIZE[(SIZE.kind == "all_trials") & (SIZE.block == "intuitive")]
print(f"\n>>> READ THIS. The intuitive block of a remapping session holds a median of only "
      f"{int(_bad.n.median())} trial bins\n    ({_bad.n.median() / SRB / 60:.1f} min of trial "
      f"time: the animal is expert on that axis and hits in ~3 s), i.e. "
      f"{_bad.n.median() / N_USED:.1f}x the neuron count.\n    A trial-bins PR there is NOT "
      f"valid and is NOT reported. The block-level PR (every in-experiment bin of the block, "
      f"{int(SIZE[(SIZE.kind == 'session') & (SIZE.block == 'intuitive')].n.median())} bins) "
      f"IS valid, and that is where\n    the intuitive-vs-new dimensionality comparison is "
      f"made — see 3.3. ITI bins never reach the floor\n    on any day, so that scenario "
      f"disappears completely.")

# ---- the sliding windows, also decided before the neural pass ----
W_BINS = int(round(SLIDE_WIN_MIN * 60 * SRB))
STEP_BINS = int(round(SLIDE_STEP_MIN * 60 * SRB))


def windows_of(db):
    """Windows that lie ENTIRELY inside one block.

    A window that straddled the rotation would mix the two decoder axes, and every measure
    here is about one axis at a time. The price is a gap of up to one window width at each
    block boundary -- visible in fig15 as a break, which is the honest shape of the data."""
    out = []
    for b in BLOCKS_OF[db.i]:
        if b not in USED_BLOCKS:
            continue
        idx = np.flatnonzero(block_bin_span(db, b))
        if len(idx) == 0:
            continue
        lo, hi = int(idx[0]), int(idx[-1]) + 1
        s0 = lo
        while s0 + W_BINS <= hi:
            m = np.zeros(db.nb, bool)
            m[s0:s0 + W_BINS] = True
            out.append((b, s0, s0 + W_BINS, int((m & db.inx).sum())))
            s0 += STEP_BINS
    return out


WIN = {i: windows_of(db) for i, db in DB.items()}
_wn = np.array([w[3] for ws in WIN.values() for w in ws])
SLIDE_MATCH = (int(max(MIN_BINS, _wn[_wn >= MIN_BINS].min())) if (_wn >= MIN_BINS).any()
               else None)
print(f"\nsliding window {SLIDE_WIN_MIN:g} min = {W_BINS} bins, step {SLIDE_STEP_MIN:g} min = "
      f"{STEP_BINS} bins -> {len(_wn)} windows over {len(WIN)} sessions, "
      f"{int((_wn >= MIN_BINS).sum())} of them valid; matched at {SLIDE_MATCH} bins.")
print(f"per session: " + ", ".join(
    f"{b} {sum(1 for w in WIN[int(REMAP[0])] if w[0] == b)}"
    for b in ["intuitive", "new"]) + " windows on a remapping day, "
      f"{len(WIN[int(STABLE[0])])} on a stable one.")

In [ ]:

import hashlib
# One pass over each session's neural data. `spks` is binned to 6 Hz and per-neuron z-scored
# (the repo-wide preprocessing), the manifold is fitted on the WHOLE session -- never inside a
# window, which would let the subspace chase the thing being measured -- and every trial is
# projected into it. Nothing is smoothed: smoothing injects shared autocorrelation, which
# inflates both the trial-pair correlation and the apparent dimensionality.
PASS_TAG = hashlib.md5(repr(
    sorted(MATCH.items()) + [("neu", N_USED), ("k", MANIFOLD_K), ("w", N_WARP),
                             ("win", SLIDE_WIN_MIN), ("step", SLIDE_STEP_MIN),
                             ("slide", SLIDE_MATCH), ("bin", BIN), ("int", INT_MAX_MIN),
                             ("seg", SEG_PARTS, SEG_CAP),
                             # the conditions themselves: adding one need not move any MATCH
                             # value, and then a stale cache would silently lack its key
                             ("cond", tuple(sorted({b for bs in PR_BLOCKS_OF.values()
                                                    for b in bs})), tuple(KINDS))]
).encode()).hexdigest()[:8]


def trial_thirds(db):
    """Label every trial bin with which SEG_PARTS-th of its own trial it falls in.

    The trial is cut in its REAL length, not warped: the question in 3.5 is whether the
    population uses more dimensions early or late in a trial, and warping would erase the
    very thing being asked about. The labels do not depend on the clustering, so re-cutting
    the clusters never forces this pass to run again."""
    bi, bt, bp = [], [], []
    for j in np.flatnonzero(db.ok):
        a, b = int(db.sb[j]), int(db.eb[j])
        if b - a < SEG_PARTS:
            continue
        e = np.round(np.linspace(a, b, SEG_PARTS + 1)).astype(int)
        for p in range(SEG_PARTS):
            if e[p + 1] > e[p]:
                r = np.arange(e[p], e[p + 1])
                bi.append(r); bt.append(np.full(len(r), j)); bp.append(np.full(len(r), p))
    if not bi:
        return (np.zeros(0, int),) * 3
    bi, bt, bp = np.concatenate(bi), np.concatenate(bt), np.concatenate(bp)
    keep = db.inx[bi]
    return bi[keep], bt[keep], bp[keep]


def neural_day(i):
    S = bmi.session(MOUSE, i)
    db = DB[i]
    fp = CACHE / f"neu_{S.day}_{PASS_TAG}.npz"
    if fp.exists() and not RECOMPUTE:
        return dict(np.load(fp, allow_pickle=True))
    XT = np.ascontiguousarray(
        na.bin_and_zscore(na.load_neural(bmi.state().R[MOUSE], S.day), BIN)[:, :db.nb].T)
    if NEU is not None:
        XT = np.ascontiguousarray(XT[:, NEU])
    spec = na.pca_spectrum(XT[db.inx])
    sub = na.fit_subspace(XT[db.inx], MANIFOLD_K)
    Y = sub.project(XT)
    n = len(db.t)
    Wt = np.full((n, N_WARP, MANIFOLD_K), np.nan, np.float32)
    L = np.full(n, np.nan)
    NBT = np.zeros(n, int)
    for j in np.flatnonzero(db.ok):
        tj = Y[db.sb[j]:db.eb[j]]
        L[j] = na.trajectory_length(tj)
        NBT[j] = len(tj)
        Wt[j] = na.resample_trajectory(tj, N_WARP).astype(np.float32)
    keys, pm, ps, po, pn = [], [], [], [], []
    for b in PR_BLOCKS_OF[i]:
        for k in KINDS:
            m = cond_mask(db, b, k)
            mu, sd, okk = pr_matched(XT[m], MATCH[k])
            keys.append(f"{b}|{k}"); pm.append(mu); ps.append(sd)
            po.append(okk); pn.append(int(m.sum()))
    wb, ws, wp, wps, wok, wn = [], [], [], [], [], []
    for (b, s0, s1, nv) in WIN[i]:
        m = np.zeros(db.nb, bool); m[s0:s1] = True; m &= db.inx
        mu, sd, okk = pr_matched(XT[m], SLIDE_MATCH)
        wb.append(b); ws.append(s0); wp.append(mu); wps.append(sd)
        wok.append(okk); wn.append(nv)
    # the thirds, capped so the pooled matrices in 3.5 stay a sane size
    bi, bt, bp = trial_thirds(db)
    if len(bi) > SEG_CAP:
        sel = np.sort(np.random.default_rng(SEED + int(i)).choice(len(bi), SEG_CAP,
                                                                  replace=False))
        bi, bt, bp = bi[sel], bt[sel], bp[sel]
    out = dict(W=Wt, L=L, NBT=NBT, evr=spec.ratio.astype(np.float32),
               var_k=float(np.sum(spec.ratio[:MANIFOLD_K])), pr_full=spec.participation_ratio,
               pr_keys=np.array(keys), pr_mean=np.array(pm, float), pr_sd=np.array(ps, float),
               pr_ok=np.array(po, bool), pr_n=np.array(pn, int),
               win_block=np.array(wb), win_s=np.array(ws, int), win_pr=np.array(wp, float),
               win_pr_sd=np.array(wps, float), win_ok=np.array(wok, bool),
               win_n=np.array(wn, int), sb=db.sb, eb=db.eb, ok=db.ok,
               seg_X=XT[bi].astype(np.float32), seg_trial=bt.astype(np.int32),
               seg_part=bp.astype(np.int8))
    np.savez_compressed(fp, **out)
    return out


_t0 = time.time()
ND = {}
for _n, i in enumerate(sorted(DB)):
    ND[i] = neural_day(i)
    if _n == 0:
        print(f"first session in {time.time() - _t0:.0f}s"
              + ("" if (CACHE / f'neu_{bmi.session(MOUSE, i).day}_{PASS_TAG}.npz').exists()
                 else " (building)"))
print(f"{len(ND)} sessions ready in {time.time() - _t0:.0f}s   [tag {PASS_TAG}]")
_vk = np.array([ND[i]["var_k"] for i in ND])
ELBOW80 = np.array([int(np.searchsorted(np.cumsum(ND[i]["evr"]), 0.80) + 1) for i in ND])
print(f"the manifold is the top {MANIFOLD_K} PCs of each session, fitted on every "
      f"in-experiment bin of that session:\n  it holds {_vk.min():.0%}–{_vk.max():.0%} of the "
      f"population variance (median {np.median(_vk):.0%}); the full-session participation "
      f"ratio is {np.min([ND[i]['pr_full'] for i in ND]):.0f}–"
      f"{np.max([ND[i]['pr_full'] for i in ND]):.0f}.")
print(f"  {MANIFOLD_K} PCs is therefore a DELIBERATELY RESTRICTED view, not the whole "
      f"manifold: 80 % of the variance\n  would need {ELBOW80.min()}–{ELBOW80.max()} PCs "
      f"(median {int(np.median(ELBOW80))}). That is the point — the correlation and the "
      f"length\n  measures are about the dominant structure, and a space that big would "
      f"mostly add noise. The slice is\n  the same size on every session, so days stay "
      f"comparable; raise MANIFOLD_K to widen it.")
print(f"  trial thirds stored for 3.5: {sum(len(ND[i]['seg_part']) for i in ND):,} bins over "
      f"{len(ND)} sessions (cap {SEG_CAP}/session).")

In [ ]:

PR_KIND_FOR = {"all": "all_trials", "hit": "hit_trials"}
# Panel A of every measure figure draws ONE PR definition: the block-level one, every
# in-experiment bin of the block, matched at MATCH["session"]. It is the only definition that
# is valid on BOTH decoder axes (the intuitive block has too few trial bins -- see 3.3), so it
# is the only one two axes can be compared with. On remapping days the new axis is taken from
# its FIRST INT_MAX_MIN minutes, so the two sides also cover the same span of task time.
PR_BLOCK_FOR = {"single": "single", "intuitive": "intuitive", "new": "new_early"}


def block_pr(i, blk, kind="session", what="pr_mean"):
    """One cached block-level PR, or NaN where that condition was never computed."""
    k = list(ND[i]["pr_keys"])
    key = f"{blk}|{kind}"
    return float(ND[i][what][k.index(key)]) if key in k else np.nan


def cond_trials(i, block, outcome=None):
    db = DB[i]
    sel = db.ok & (db.t.block.to_numpy() == block) & db.t.use.to_numpy()
    if outcome is not None:
        sel &= (db.t.outcome.to_numpy() == outcome)
    return np.flatnonzero(sel)


def measures_for(i, block, which="all"):
    """The three measures for one (session, block, trial subset)."""
    nd, j = ND[i], cond_trials(i, block, None if which == "all" else which)
    r = pair_corr(nd["W"][j])
    cm, clo, chi, cn = mean_ci(r)
    lm, llo, lhi, ln = mean_ci(nd["L"][j])
    # distance per bin = speed through the manifold. It is NOT plotted anywhere -- it only
    # appears in the paired printout below, because raw length is dominated by how long the
    # trial lasted (r = 0.96) and this is the part of it that is not duration.
    rm_, rlo, rhi, _ = mean_ci(nd["L"][j] / np.maximum(nd["NBT"][j], 1))
    key = f"{block}|{PR_KIND_FOR[which]}"
    kk = list(nd["pr_keys"])
    q = kk.index(key) if key in kk else None
    pb = PR_BLOCK_FOR[block]
    return dict(i=i, rel=rel(i), block=block, which=which, n_trials=len(j),
                pr=block_pr(i, pb), pr_sd=block_pr(i, pb, what="pr_sd"),
                pr_ok=bool(np.isfinite(block_pr(i, pb))), pr_src=pb,
                pr_trial=(nd["pr_mean"][q] if q is not None else np.nan),
                pr_trial_ok=(bool(nd["pr_ok"][q]) if q is not None else False),
                pr_trial_bins=(int(nd["pr_n"][q]) if q is not None else 0),
                pr_block_whole=block_pr(i, block),
                corr=cm, corr_lo=clo, corr_hi=chi, n_pairs=len(r),
                len=lm, len_lo=llo, len_hi=lhi, n_len=ln,
                rate=rm_, rate_lo=rlo, rate_hi=rhi)


MEAS = pd.DataFrame([measures_for(int(i), b, w)
                     for i in sorted(DB) for b in BLOCKS_OF[int(i)] if b in USED_BLOCKS
                     for w in ("all", "hit")])


def draw_measure(ax, x, m, lo, hi, color, *, band=True, sd=None, label=None, ls="-"):
    """One measure series: a line through the means, with its 95 % interval (or bootstrap sd
    for the PR, which is one number per condition rather than a distribution over trials)."""
    x, m = np.asarray(x, float), np.asarray(m, float)
    g = np.isfinite(m)
    if band and lo is not None:
        ax.fill_between(x[g], np.asarray(lo, float)[g], np.asarray(hi, float)[g],
                        color=color, alpha=0.22, lw=0, zorder=2)
    if sd is not None:
        ax.errorbar(x[g], m[g], yerr=np.asarray(sd, float)[g], fmt="none", ecolor=color,
                    elinewidth=1.3, capsize=2.5, alpha=0.8, zorder=3)
    ax.plot(x[g], m[g], "o" + ls, color=color, ms=5, lw=1.9, zorder=4, label=label)
    return int(g.sum())


def measure_frame(ax, key, *, xlabel="session", tint_title=True):
    ax.set_ylabel(MEASURES[key]["label"], fontsize=12.5 * FONT_SCALE,
                  color=(MEASURES[key]["c"] if tint_title else P["fg"]))
    ax.set_xlabel(xlabel)
    ax.grid(axis="y", alpha=0.3)


print(f"trial-pair correlation: every pair is used exactly up to {PAIR_EXACT_MAX} trials, "
      f"which covers every condition here\n  (largest = {int(MEAS.n_trials.max())} trials = "
      f"{int(MEAS.n_pairs.max()):,} pairs), so no pair sampling is needed anywhere.")
print(f"PR in every panel = block level, matched at {MATCH['session']} bins "
      f"({MATCH['session'] / N_USED:.1f}x); on a remapping day the new axis is its first "
      f"{INT_MAX_MIN:g} min\n  ('new_early'), so the two axes match in bin count AND in span. "
      f"3.3 prints every other scenario.")
_d = pd.concat([pd.DataFrame(dict(dur=DB[i].t.dur.to_numpy()[cond_trials(i, b)],
                                  L=ND[i]['L'][cond_trials(i, b)],
                                  nb=ND[i]['NBT'][cond_trials(i, b)]))
                for i in sorted(DB) for b in BLOCKS_OF[int(i)] if b in USED_BLOCKS])
_d = _d[np.isfinite(_d.L)]
_rate = _d.L / np.maximum(_d.nb, 1)
print(f">>> trajectory length vs trial duration: r = {np.corrcoef(_d.dur, _d.L)[0, 1]:.2f} "
      f"over {len(_d)} trials.\n    Length is deliberately NOT warped, so a longer trial "
      f"travels further, and at that correlation the raw\n    length is close to a "
      f"restatement of the trial duration. The duration-free version is LENGTH PER BIN\n    "
      f"(speed through the manifold), whose correlation with duration is only "
      f"r = {np.corrcoef(_d.dur, _rate)[0, 1]:+.2f}; it is\n    kept out of the figures and "
      f"reported in the paired tables below, where it is the number to trust.")

### 3.1 · All trials (hit and miss)

In [ ]:

FIGNAME, WHICH, TITLE = "fig09_measures_all_trials_stable", "all", "all trials (hit and miss)"
d = MEAS[(MEAS.which == WHICH) & (MEAS.block == "single")].sort_values("rel")
fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.4), constrained_layout=True)
draw_measure(axes[0], d.rel, d.pr, None, None, C_PR, band=False, sd=d.pr_sd)
draw_measure(axes[1], d.rel, d["corr"], d.corr_lo, d.corr_hi, C_CORR)
draw_measure(axes[2], d.rel, d["len"], d.len_lo, d.len_hi, C_LEN)
for ax, k, letter in zip(axes, ["pr", "corr", "len"], "abc"):
    measure_frame(ax, k)
    dw.label_panel(ax, letter, color=P["fg"])
axes[0].set_title(f"block level, matched at {MATCH['session']} bins, {PR_BOOT} draws",
                  fontsize=11.5 * FONT_SCALE, color=P["muted"])
axes[1].set_title(f"{int(d.n_pairs.sum()):,} pairs, all of them", fontsize=11.5 * FONT_SCALE,
                  color=P["muted"])
axes[2].set_title(f"{int(d.n_len.sum()):,} trials, unwarped", fontsize=11.5 * FONT_SCALE,
                  color=P["muted"])
fig.suptitle(f"Stable control · {TITLE}   (bars = bootstrap sd, bands = 95 % on the mean)",
             fontsize=15.5 * FONT_SCALE)
st.save_fig(fig, FIGNAME); plt.show()
for k, c in [("pr", "pr"), ("corr", "corr"), ("len", "len")]:
    v = d[c].to_numpy()
    g = np.isfinite(v)
    if g.sum() > 2:
        rho = np.corrcoef(d.rel.to_numpy()[g], v[g])[0, 1]
        print(f"{MEASURES[k]['short']:>7s}: {v[g][0]:.3g} -> {v[g][-1]:.3g} across sessions "
              f"{int(d.rel.to_numpy()[g][0])}-{int(d.rel.to_numpy()[g][-1])},  r with session "
              f"= {rho:+.2f}")

In [ ]:

FIGNAME, WHICH, TITLE = ("fig10_measures_all_trials_remapping", "all", "all trials (hit and miss)")
# Panel A is ONE definition on both axes: the block-level PR, with the new axis taken from its
# first INT_MAX_MIN minutes so that the two sides match in span as well as in bin count. The
# trial-bins PR that used to sit here exists on the new axis only -- the intuitive block never
# reaches the 5x floor -- and a panel with one axis in it cannot answer a two-axis question.
# The legend is a figure-level strip above the row, so all three panels keep the same width.
fig, axes = plt.subplots(1, 3, figsize=(16.6, 4.8), constrained_layout=True)
for blk, col, nm in [("intuitive", C_INT, "intuitive axis"), ("new", C_NEW, "new axis")]:
    d = MEAS[(MEAS.which == WHICH) & (MEAS.block == blk)].sort_values("rel")
    draw_measure(axes[0], d.rel, d.pr, None, None, col, band=False, sd=d.pr_sd, label=nm)
    draw_measure(axes[1], d.rel, d["corr"], d.corr_lo, d.corr_hi, col, label=nm)
    draw_measure(axes[2], d.rel, d["len"], d.len_lo, d.len_hi, col, label=nm)
for ax, k, letter in zip(axes, ["pr", "corr", "len"], "abc"):
    measure_frame(ax, k)
    dw.label_panel(ax, letter, color=P["fg"])
axes[0].set_title(f"block level, matched at {MATCH['session']} bins · new axis = its first "
                  f"{INT_MAX_MIN:g} min", fontsize=10.5 * FONT_SCALE, color=P["muted"])
axes[1].set_title(f"{int(MEAS[(MEAS.which == WHICH) & MEAS.block.isin(['intuitive', 'new'])].n_pairs.sum()):,}"
                  f" pairs, all of them", fontsize=11.5 * FONT_SCALE, color=P["muted"])
axes[2].set_title("unwarped; see the printout for the duration-free version",
                  fontsize=10.5 * FONT_SCALE, color=P["muted"])
fig.legend(handles=[Line2D([], [], color=C_INT, lw=2.6, marker="o", ms=6,
                           label="intuitive axis (first 10 min)"),
                    Line2D([], [], color=C_NEW, lw=2.6, marker="o", ms=6,
                           label="new axis")],
           loc="outside lower center", ncol=2, frameon=False, fontsize=11 * FONT_SCALE)
fig.suptitle(f"Remapping sessions · {TITLE}   (one point per axis per session; the washout is "
             f"excluded)", fontsize=15.5 * FONT_SCALE)
st.save_fig(fig, FIGNAME); plt.show()
for c, k, nm in [("corr", "corr", ""), ("len", "len", ""), ("rate", "len", " per bin"),
                 ("pr", "pr", " (block, span-matched)"),
                 ("pr_block_whole", "pr", " (block, whole)")]:
    a = MEAS[(MEAS.which == WHICH) & (MEAS.block == "intuitive")][c].to_numpy()
    b = MEAS[(MEAS.which == WHICH) & (MEAS.block == "new")][c].to_numpy()
    g = np.isfinite(a) & np.isfinite(b)
    if g.sum() >= 5:
        w = stats.wilcoxon(a[g], b[g])
        print(f"{(MEASURES[k]['short'] + nm):>26s}: intuitive {np.median(a[g]):.3g} vs new "
              f"{np.median(b[g]):.3g} (medians over {int(g.sum())} paired sessions), "
              f"Wilcoxon p = {w.pvalue:.3g}")
    else:
        print(f"{(MEASURES[k]['short'] + nm):>26s}: only {int(g.sum())} sessions have both "
              f"axes with this definition — no paired test")
_t = MEAS[(MEAS.which == WHICH) & (MEAS.block == "intuitive")]
print(f"\nthe trial-bins PR is still refused on the intuitive axis: median "
      f"{int(_t.pr_trial_bins.median())} bins = {_t.pr_trial_bins.median() / N_USED:.1f}x the "
      f"neuron count,\nbelow the {PR_MIN_RATIO:g}x floor. That is why panel A uses the "
      f"block-level definition on both axes.")

### 3.2 · Hit trials only

In [ ]:

FIGNAME, WHICH, TITLE = "fig11_measures_hit_trials_stable", "hit", "hit trials only"
d = MEAS[(MEAS.which == WHICH) & (MEAS.block == "single")].sort_values("rel")
fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.4), constrained_layout=True)
draw_measure(axes[0], d.rel, d.pr, None, None, C_PR, band=False, sd=d.pr_sd)
draw_measure(axes[1], d.rel, d["corr"], d.corr_lo, d.corr_hi, C_CORR)
draw_measure(axes[2], d.rel, d["len"], d.len_lo, d.len_hi, C_LEN)
for ax, k, letter in zip(axes, ["pr", "corr", "len"], "abc"):
    measure_frame(ax, k)
    dw.label_panel(ax, letter, color=P["fg"])
axes[0].set_title(f"block level, matched at {MATCH['session']} bins, {PR_BOOT} draws",
                  fontsize=11.5 * FONT_SCALE, color=P["muted"])
axes[1].set_title(f"{int(d.n_pairs.sum()):,} pairs, all of them", fontsize=11.5 * FONT_SCALE,
                  color=P["muted"])
axes[2].set_title(f"{int(d.n_len.sum()):,} trials, unwarped", fontsize=11.5 * FONT_SCALE,
                  color=P["muted"])
fig.suptitle(f"Stable control · {TITLE}   (bars = bootstrap sd, bands = 95 % on the mean)",
             fontsize=15.5 * FONT_SCALE)
st.save_fig(fig, FIGNAME); plt.show()
for k, c in [("pr", "pr"), ("corr", "corr"), ("len", "len")]:
    v = d[c].to_numpy()
    g = np.isfinite(v)
    if g.sum() > 2:
        rho = np.corrcoef(d.rel.to_numpy()[g], v[g])[0, 1]
        print(f"{MEASURES[k]['short']:>7s}: {v[g][0]:.3g} -> {v[g][-1]:.3g} across sessions "
              f"{int(d.rel.to_numpy()[g][0])}-{int(d.rel.to_numpy()[g][-1])},  r with session "
              f"= {rho:+.2f}")

In [ ]:

FIGNAME, WHICH, TITLE = ("fig12_measures_hit_trials_remapping", "hit", "hit trials only")
# Panel A is ONE definition on both axes: the block-level PR, with the new axis taken from its
# first INT_MAX_MIN minutes so that the two sides match in span as well as in bin count. The
# trial-bins PR that used to sit here exists on the new axis only -- the intuitive block never
# reaches the 5x floor -- and a panel with one axis in it cannot answer a two-axis question.
# The legend is a figure-level strip above the row, so all three panels keep the same width.
fig, axes = plt.subplots(1, 3, figsize=(16.6, 4.8), constrained_layout=True)
for blk, col, nm in [("intuitive", C_INT, "intuitive axis"), ("new", C_NEW, "new axis")]:
    d = MEAS[(MEAS.which == WHICH) & (MEAS.block == blk)].sort_values("rel")
    draw_measure(axes[0], d.rel, d.pr, None, None, col, band=False, sd=d.pr_sd, label=nm)
    draw_measure(axes[1], d.rel, d["corr"], d.corr_lo, d.corr_hi, col, label=nm)
    draw_measure(axes[2], d.rel, d["len"], d.len_lo, d.len_hi, col, label=nm)
for ax, k, letter in zip(axes, ["pr", "corr", "len"], "abc"):
    measure_frame(ax, k)
    dw.label_panel(ax, letter, color=P["fg"])
axes[0].set_title(f"block level, matched at {MATCH['session']} bins · new axis = its first "
                  f"{INT_MAX_MIN:g} min", fontsize=10.5 * FONT_SCALE, color=P["muted"])
axes[1].set_title(f"{int(MEAS[(MEAS.which == WHICH) & MEAS.block.isin(['intuitive', 'new'])].n_pairs.sum()):,}"
                  f" pairs, all of them", fontsize=11.5 * FONT_SCALE, color=P["muted"])
axes[2].set_title("unwarped; see the printout for the duration-free version",
                  fontsize=10.5 * FONT_SCALE, color=P["muted"])
fig.legend(handles=[Line2D([], [], color=C_INT, lw=2.6, marker="o", ms=6,
                           label="intuitive axis (first 10 min)"),
                    Line2D([], [], color=C_NEW, lw=2.6, marker="o", ms=6,
                           label="new axis")],
           loc="outside lower center", ncol=2, frameon=False, fontsize=11 * FONT_SCALE)
fig.suptitle(f"Remapping sessions · {TITLE}   (one point per axis per session; the washout is "
             f"excluded)", fontsize=15.5 * FONT_SCALE)
st.save_fig(fig, FIGNAME); plt.show()
for c, k, nm in [("corr", "corr", ""), ("len", "len", ""), ("rate", "len", " per bin"),
                 ("pr", "pr", " (block, span-matched)"),
                 ("pr_block_whole", "pr", " (block, whole)")]:
    a = MEAS[(MEAS.which == WHICH) & (MEAS.block == "intuitive")][c].to_numpy()
    b = MEAS[(MEAS.which == WHICH) & (MEAS.block == "new")][c].to_numpy()
    g = np.isfinite(a) & np.isfinite(b)
    if g.sum() >= 5:
        w = stats.wilcoxon(a[g], b[g])
        print(f"{(MEASURES[k]['short'] + nm):>26s}: intuitive {np.median(a[g]):.3g} vs new "
              f"{np.median(b[g]):.3g} (medians over {int(g.sum())} paired sessions), "
              f"Wilcoxon p = {w.pvalue:.3g}")
    else:
        print(f"{(MEASURES[k]['short'] + nm):>26s}: only {int(g.sum())} sessions have both "
              f"axes with this definition — no paired test")
_t = MEAS[(MEAS.which == WHICH) & (MEAS.block == "intuitive")]
print(f"\nthe trial-bins PR is still refused on the intuitive axis: median "
      f"{int(_t.pr_trial_bins.median())} bins = {_t.pr_trial_bins.median() / N_USED:.1f}x the "
      f"neuron count,\nbelow the {PR_MIN_RATIO:g}x floor. That is why panel A uses the "
      f"block-level definition on both axes.")


### 3.3 · Which bins the PR is estimated from

No figure — just the table behind the choice panel A makes. Within a scenario the bin count is
matched, so the conditions are comparable; across scenarios they are not.

In [ ]:

# Every way of choosing WHICH bins the participation ratio is estimated from, in one table.
# Within a scenario the match count is identical, so the conditions are comparable; ACROSS
# scenarios they are not, because each is matched at its own bin count.
GROUPS = [("stable control", "single"), ("intuitive (10 min)", "intuitive"),
          ("new, first 10 min", "new_early"), ("new, whole block", "new")]


def scenario_prs(blk, kind):
    v = np.array([ND[i]["pr_mean"][list(ND[i]["pr_keys"]).index(f"{blk}|{kind}")]
                  for i in sorted(DB) if f"{blk}|{kind}" in list(ND[i]["pr_keys"])])
    return v[np.isfinite(v)]


print(f"participation ratio, median over sessions   [{N_USED} neurons, floor = "
      f"{PR_MIN_RATIO:g}x = {MIN_BINS} bins]\n")
print(f"{'scenario':<14s}" + "".join(f"{nm:>22s}" for nm, _ in GROUPS))
for kind in KINDS:
    if MATCH[kind] is None:
        print(f"{kind:<14s}" + "  DROPPED — no session reaches the floor "
              f"(largest anywhere = {int(SIZE[SIZE.kind == kind].n.max())} bins)")
        continue
    cells = []
    for _nm, blk in GROUPS:
        v = scenario_prs(blk, kind)
        cells.append(f"{np.median(v):.1f} (n={len(v)})" if len(v) else "not reported")
    print(f"{kind:<14s}" + "".join(f"{c:>22s}" for c in cells)
          + f"    matched at {MATCH[kind]} bins ({MATCH[kind] / N_USED:.1f}x)")

_p = {b: np.array([ND[i]["pr_mean"][list(ND[i]["pr_keys"]).index(f"{b}|session")]
                   for i in REMAP]) for b in ("intuitive", "new", "new_early")}
_g = np.all([np.isfinite(v) for v in _p.values()], axis=0)
print(f"\nblock-level PR, paired within a remapping session ({int(_g.sum())} sessions):")
print(f"  intuitive (10 min)       {np.median(_p['intuitive'][_g]):5.1f}")
print(f"  new axis, first 10 min   {np.median(_p['new_early'][_g]):5.1f}   vs intuitive: "
      f"Wilcoxon p = {stats.wilcoxon(_p['intuitive'][_g], _p['new_early'][_g]).pvalue:.3g}"
      f"   <-- SPAN-MATCHED, and what panel A of fig10/fig12 draws")
print(f"  new axis, whole block    {np.median(_p['new'][_g]):5.1f}   vs intuitive: "
      f"Wilcoxon p = {stats.wilcoxon(_p['intuitive'][_g], _p['new'][_g]).pvalue:.3g}")
print(f"\n>>> The whole-block number is INFLATED. Matching the bin COUNT does not match the "
      f"time SPAN those\n    bins are drawn from: the new block runs ~"
      f"{np.median([(block_frames(int(i), 'new')[1] - block_frames(int(i), 'new')[0]) / SR / 60 for i in REMAP]):.0f}"
      f" min against the intuitive block's {INT_MAX_MIN:g}, and slow drift across an\n    "
      f"hour adds dimensions by itself. The first-10-min column removes that, and fig16 "
      f"(matched\n    {SLIDE_WIN_MIN:g} min windows throughout) is the within-session version "
      f"of the same control.")


### 3.4 · Within a session: a sliding window

A window never crosses a block boundary, so the two decoder axes are never mixed. The cost is
a gap either side of the rotation, which is left visible. Sessions are averaged into bundles —
`STABLE_BUNDLE` / `REMAP_BUNDLE` sessions per line — because one line per session is
unreadable past about five of them.

In [ ]:

MIN_WIN_TRIALS = 5             # a window with fewer trials gives no correlation / length


def window_table(i):
    """The three measures inside each window. The PR came from the neural pass (it needs the
    full neuron space); the correlation and the length are rebuilt here from the cached
    trajectories, so they can be retuned without touching the neural data again.

    `t_mid` is minutes into the session, `t_blk` minutes into the window's own BLOCK, and
    `t_rot` minutes from the rotation. Sessions share the second and third of those; they do
    not share the first, because a block starts where it starts."""
    nd, db, S = ND[i], DB[i], bmi.session(MOUSE, i)
    t0 = (S.remap / SR / 60.0) if np.isfinite(S.remap or np.nan) else np.nan
    lo_of = {b: float(np.flatnonzero(block_bin_span(db, b))[0])
             for b in {str(x) for x in nd["win_block"]}}
    rows = []
    for q, (b, s0) in enumerate(zip(nd["win_block"], nd["win_s"])):
        s0, b = int(s0), str(b)
        j = np.flatnonzero(db.ok & (db.sb >= s0) & (db.sb < s0 + W_BINS))
        if len(j) >= MIN_WIN_TRIALS:
            cm, clo, chi, _ = mean_ci(pair_corr(nd["W"][j]))
            lm, llo, lhi, _ = mean_ci(nd["L"][j])
        else:
            cm = clo = chi = lm = llo = lhi = np.nan
        mid = (s0 + W_BINS / 2) / SRB / 60.0
        rows.append(dict(i=i, rel=rel(i), block=b, s=s0, t_mid=mid, t_rot=mid - t0,
                         t_blk=(s0 - lo_of[b] + W_BINS / 2) / SRB / 60.0,
                         pr=float(nd["win_pr"][q]), pr_sd=float(nd["win_pr_sd"][q]),
                         pr_ok=bool(nd["win_ok"][q]), corr=cm, corr_lo=clo, corr_hi=chi,
                         len=lm, len_lo=llo, len_hi=lhi, n_trials=len(j)))
    return pd.DataFrame(rows)


WINTAB = pd.concat([window_table(int(i)) for i in sorted(DB)], ignore_index=True)
BCOL = {"single": P["muted"], "intuitive": C_INT, "new": C_NEW}


def bundles(idxs, size, n_max):
    """Consecutive groups of `size` sessions, at most `n_max` of them. size = 1 gives one
    group per session, which is the old one-line-per-day figure."""
    size = max(int(size), 1)
    g = [list(idxs)[k:k + size] for k in range(0, len(idxs), size)]
    return [b for b in g if b][:int(n_max)]


def bundle_label(grp, idxs, word="sessions"):
    r = [list(idxs).index(i) + 1 for i in grp]
    return f"{word} {r[0]}" if len(r) == 1 else f"{word} {r[0]}–{r[-1]}"


def bundle_curve(d, xcol, key, step):
    """Several sessions as one line: the mean of `key` over `step`-minute bins of `xcol`,
    with the SEM across the windows that land in each bin. Binning is what makes sessions
    poolable at all -- their window grids start at their own block boundaries."""
    v = d[np.isfinite(d[key]) & np.isfinite(d[xcol])].copy()
    if not len(v):
        return None
    v["_b"] = np.round(v[xcol] / step).astype(int)
    g = v.groupby("_b")[key]
    n = g.size().to_numpy()
    return (g.size().index.to_numpy() * step, g.mean().to_numpy(),
            np.where(n > 1, g.std(ddof=1).to_numpy() / np.sqrt(np.maximum(n, 1)), np.nan), n)


print(f"{len(WINTAB)} windows: {int(WINTAB.pr_ok.sum())} with a valid PR, "
      f"{int(WINTAB['corr'].notna().sum())} with enough trials "
      f"(>= {MIN_WIN_TRIALS}) for a correlation.")
_r = WINTAB[WINTAB.block.isin(["intuitive", "new"])]
print(f"on a remapping session: {int(_r[_r.block == 'intuitive'].groupby('i').size().median())}"
      f" windows on the intuitive axis and "
      f"{int(_r[_r.block == 'new'].groupby('i').size().median())} on the new one (medians).")
print(f"the {SLIDE_WIN_MIN:g} min window never crosses the rotation, so the two axes are never "
      f"mixed;\nthe cost is a gap of up to {SLIDE_WIN_MIN:g} min on each side of the boundary, "
      f"left visible rather than filled.")
print(f"bundles: stable {STABLE_BUNDLE} session(s) per line x {STABLE_N_BUNDLES} lines, "
      f"remapping {REMAP_BUNDLE} x {REMAP_N_BUNDLES}.")

In [ ]:

FIGNAME = "fig14_sliding_window_stable_bundles"
# Consecutive stable-control sessions averaged into one line each, on the same day ramp as
# every other across-session figure. The measure colour survives only on the y label, which is
# the one place it is not competing with the day colour for the same ink.
GRPS = bundles(STABLE, STABLE_BUNDLE, STABLE_N_BUNDLES)
cols = st.day_colors(len(GRPS), cmap=DAY_CMAP)
fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.6), constrained_layout=True)
for ax, key, letter in zip(axes, ["pr", "corr", "len"], "abc"):
    for col, grp in zip(cols, GRPS):
        d = WINTAB[WINTAB.i.isin(grp)]
        if key == "pr":
            d = d[d.pr_ok]
        cur = bundle_curve(d, "t_blk", key, SLIDE_STEP_MIN)
        if cur is None:
            continue
        x, m, s, n = cur
        ax.fill_between(x, m - s, m + s, color=col, alpha=0.20, lw=0, zorder=2)
        ax.plot(x, m, "o-", color=col, ms=5, lw=2.0, zorder=3,
                label=bundle_label(grp, STABLE))
    measure_frame(ax, key, xlabel="time into the session (min)")
    dw.label_panel(ax, letter, color=P["fg"])
axes[0].legend(fontsize=10.5 * FONT_SCALE, loc="best", title="mean of", title_fontsize=9.5)
fig.suptitle(f"Stable control, {SLIDE_WIN_MIN:g} min windows stepped by "
             f"{SLIDE_STEP_MIN:g} min   (all trials, {STABLE_BUNDLE} session(s) per line)",
             fontsize=15.5 * FONT_SCALE)
st.save_fig(fig, FIGNAME); plt.show()
print("bundles: " + "   ".join(f"{bundle_label(g, STABLE)} = sessions "
                               f"{', '.join(str(rel(int(i))) for i in g)}" for g in GRPS))
print(f"the manifold is the whole session's, not the window's — a window-local PCA would "
      f"re-centre on\nwhatever the animal was doing at that moment and hide exactly the drift "
      f"these panels are for.\nBands are the SEM across the windows pooled into each "
      f"{SLIDE_STEP_MIN:g} min bin, not across trials.")

In [ ]:

# One figure per remapping session you want to look at -- set SLIDE_REMAP_RANKS at the top.
for _rank in SLIDE_REMAP_RANKS:
    if _rank >= len(REMAP):
        continue
    i = int(REMAP[_rank])
    FIGNAME = f"fig15_sliding_window_remap{_rank + 1:02d}_session{rel(i)}"
    d_all = WINTAB[WINTAB.i == i]
    fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.4), constrained_layout=True)
    for ax, key, letter in zip(axes, ["pr", "corr", "len"], "abc"):
        for blk in ["intuitive", "new"]:
            d = d_all[d_all.block == blk].sort_values("t_rot")
            if not len(d):
                continue
            if key == "pr":
                d = d[d.pr_ok]
            draw_measure(ax, d.t_rot, d[key],
                         None if key == "pr" else d[f"{key}_lo"],
                         None if key == "pr" else d[f"{key}_hi"], BCOL[blk],
                         band=(key != "pr"), sd=(d.pr_sd if key == "pr" else None),
                         label=f"{blk} axis")
        ax.axvline(0.0, color=P["fg"], ls="--", lw=1.4, zorder=1)
        measure_frame(ax, key, xlabel="window centre, min from the rotation")
        dw.label_panel(ax, letter, color=P["fg"])
    dw.legend_from(axes[1], [("intuitive axis", "line", C_INT, dict(lw=2.6, marker="o", ms=6)),
                             ("new axis", "line", C_NEW, dict(lw=2.6, marker="o", ms=6))],
                   loc="best", fontsize=11 * FONT_SCALE)
    fig.suptitle(f"Session {rel(i)} ({bmi.session(MOUSE, i).day}) — remapping day "
                 f"{_rank + 1}, {SLIDE_WIN_MIN:g} min windows   (all trials)",
                 fontsize=15.5 * FONT_SCALE)
    st.save_fig(fig, FIGNAME); plt.show()
print(f"the dashed line is the rotation. No window spans it: the nearest window centres sit "
      f"about\n{SLIDE_WIN_MIN / 2:g} min either side, which is the boundary cost of refusing "
      f"to mix the two axes.")

In [ ]:

FIGNAME = "fig16_sliding_window_remap_bundles"
# Every remapping session, but bundled. Each session's own INTUITIVE-block mean is subtracted
# FIRST, so what is averaged is the change the rotation produced rather than the day's
# baseline level -- that subtraction is what makes two days comparable at all. The two blocks
# are drawn as separate lines, so nothing is interpolated across the window gap at the
# rotation.
GRPS = bundles(REMAP, REMAP_BUNDLE, REMAP_N_BUNDLES)
cols = st.day_colors(len(GRPS), cmap=DAY_CMAP)
D16 = []
for i in REMAP:
    d = WINTAB[WINTAB.i == int(i)].copy()
    for key in ["pr", "corr", "len"]:
        ok = d.pr_ok if key == "pr" else np.isfinite(d[key])
        b = d.loc[ok & (d.block == "intuitive"), key]
        d[f"d_{key}"] = (d[key] - float(b.mean())) if len(b) else np.nan
    D16.append(d)
D16 = pd.concat(D16, ignore_index=True)

fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.8), constrained_layout=True)
for ax, key, letter in zip(axes, ["pr", "corr", "len"], "abc"):
    for col, grp in zip(cols, GRPS):
        for blk in ("intuitive", "new"):
            d = D16[D16.i.isin(grp) & (D16.block == blk)]
            if key == "pr":
                d = d[d.pr_ok]
            cur = bundle_curve(d, "t_rot", f"d_{key}", SLIDE_STEP_MIN)
            if cur is None:
                continue
            x, m, s, n = cur
            ax.fill_between(x, m - s, m + s, color=col, alpha=0.18, lw=0, zorder=2)
            ax.plot(x, m, "o-", color=col, ms=4.5, lw=2.0, zorder=3,
                    label=(bundle_label(grp, REMAP, "remapping days") if blk == "new"
                           else None))
    ax.axvline(0.0, color=P["fg"], ls="--", lw=1.4, zorder=1)
    ax.axhline(0.0, color=P["muted"], ls=":", lw=1.2, zorder=1)
    ax.set_ylabel(f"Δ {MEASURES[key]['label']}\n(re this session's intuitive axis)",
                  fontsize=11.5 * FONT_SCALE, color=MEASURES[key]["c"])
    ax.set_xlabel("window centre, min from the rotation")
    ax.grid(axis="y", alpha=0.3)
    dw.label_panel(ax, letter, color=P["fg"])
fig.legend(handles=[Line2D([], [], color=c, lw=2.4, marker="o", ms=5,
                           label=bundle_label(g, REMAP, "days"))
                    for c, g in zip(cols, GRPS)],
           loc="outside lower center", ncol=min(len(GRPS), 5), frameon=False,
           title="mean of remapping", fontsize=10.5 * FONT_SCALE)
fig.suptitle(f"Remapping sessions, baseline-subtracted   (all trials, {REMAP_BUNDLE} "
             f"session(s) per line)", fontsize=15.5 * FONT_SCALE)
st.save_fig(fig, FIGNAME); plt.show()

print("bundles: " + "   ".join(f"{bundle_label(g, REMAP, 'days')} = sessions "
                               f"{', '.join(str(rel(int(i))) for i in g)}" for g in GRPS))
for key in ["pr", "corr", "len"]:
    rows = []
    for i in REMAP:
        d = WINTAB[(WINTAB.i == int(i))]
        d = d[d.pr_ok] if key == "pr" else d
        rows.append((d.loc[d.block == "intuitive", key].mean(),
                     d.loc[(d.block == "new") & (d.t_rot <= SLIDE_WIN_MIN + SLIDE_STEP_MIN),
                           key].mean()))
    a, b = np.array([r[0] for r in rows]), np.array([r[1] for r in rows])
    g = np.isfinite(a) & np.isfinite(b)
    if g.sum() >= 5:
        print(f"{MEASURES[key]['short']:>7s}: first windows after the rotation minus the "
              f"intuitive mean = {np.median(b[g] - a[g]):+.3g} "
              f"(median over {int(g.sum())} sessions), Wilcoxon p = "
              f"{stats.wilcoxon(a[g], b[g]).pvalue:.3g}")


### 3.5 · Early, middle and late inside a trial

Every trial is cut into three equal parts of its own **unwarped** length. All the early bins of
one cluster are pooled, then all the middle bins, then all the late ones, and each pool gets
its own participation ratio. Pooling is across sessions — a single session holds about twice
the neuron count in a third of its trials, nowhere near the 5× floor.

In [ ]:

FIGNAME = "fig19_pr_early_middle_late_by_cluster"
PART_NAMES = (["early", "middle", "late"] if SEG_PARTS == 3
              else [f"part {p + 1}" for p in range(SEG_PARTS)])


def seg_arrays(oc, kk, part):
    """The stored bins of one (outcome, cluster, third), ONE ARRAY PER SESSION.

    Kept per session rather than stacked, so the bootstrap below can resample whole sessions:
    bins inside a session are nothing like independent, and a bin-level bootstrap would give
    an interval several times too narrow."""
    sub = CLU[oc]["sub"]
    out = []
    for i in sorted(DB):
        tr = sub.loc[(sub.i == i) & (sub.cluster == kk), "trial"].to_numpy()
        if not len(tr):
            continue
        nd = ND[i]
        m = (nd["seg_part"] == part) & np.isin(nd["seg_trial"], tr)
        if m.any():
            out.append(nd["seg_X"][m])
    return out


def seg_pr(arrs, n_match, n_boot=SEG_BOOT, seed=SEED):
    """PR of the pooled bins at a FIXED sample size, plus a bootstrap over sessions.

    Every session was z-scored on its own before pooling, and the three thirds are pooled the
    same way over the same sessions, so whatever between-session structure survives is common
    to all three and cannot by itself make early differ from late."""
    rng = np.random.default_rng(seed)
    X = np.vstack(arrs)
    if len(X) < n_match or n_match < PR_MIN_RATIO * N_USED:
        return np.nan, np.array([])
    point = pr_of(X[rng.choice(len(X), n_match, replace=False)])
    draws = []
    for _ in range(n_boot):
        s = rng.integers(0, len(arrs), len(arrs))
        Xb = np.vstack([arrs[k] for k in s])
        if len(Xb) >= n_match:
            draws.append(pr_of(Xb[rng.choice(len(Xb), n_match, replace=False)]))
    return point, np.array(draws)


_t0 = time.time()
SEGS = {(oc, int(kk), p): seg_arrays(oc, int(kk), p)
        for oc in ["hit", "miss"] for kk in np.unique(CLU[oc]["labels"])
        for p in range(SEG_PARTS)}
SEG_N = {k: int(sum(len(a) for a in v)) for k, v in SEGS.items()}
# Matched WITHIN a cluster, to its own smallest third. The question is early vs middle vs late
# inside one cluster, and that is the comparison this protects. Matching every cluster of an
# outcome to one number instead would let the smallest cluster veto all the others -- miss C2
# holds 80 trials and would have taken miss C1 down with it. The price is that two clusters
# are estimated at different sample sizes and so cannot be read against each other.
SEG_MATCH = {}
for oc in ["hit", "miss"]:
    for kk in np.unique(CLU[oc]["labels"]):
        ns = [SEG_N[(oc, int(kk), p)] for p in range(SEG_PARTS)]
        SEG_MATCH[(oc, int(kk))] = int(min(ns)) if min(ns) >= MIN_BINS else None
SEG_PR = {k: (seg_pr(v, SEG_MATCH[(k[0], k[1])]) if (SEG_MATCH[(k[0], k[1])] and v)
              else (np.nan, np.array([]))) for k, v in SEGS.items()}
print(f"segment PRs in {time.time() - _t0:.0f}s")

fig, axes = plt.subplots(1, 2, figsize=(14.0, 4.9), constrained_layout=True)
for ax, oc in zip(axes, ["hit", "miss"]):
    ks = np.unique(CLU[oc]["labels"])
    if all(SEG_MATCH[(oc, int(k))] is None for k in ks):
        ax.text(0.5, 0.5, f"{oc}\n\nno cluster reaches {MIN_BINS} bins in every third\n"
                          f"({PR_MIN_RATIO:g} × {N_USED} neurons)\n\nnot reported",
                transform=ax.transAxes, ha="center", va="center", color=C_BAD,
                fontsize=12 * FONT_SCALE)
        ax.set_xticks([]); ax.set_title(oc, fontsize=13 * FONT_SCALE); continue
    pos, lab = [], []
    for j, kk in enumerate(ks):
        if SEG_MATCH[(oc, int(kk))] is None:
            ax.text(j * (SEG_PARTS + 1) + (SEG_PARTS - 1) / 2, 0.5,
                    f"C{kk}\n{min(SEG_N[(oc, int(kk), p)] for p in range(SEG_PARTS)):,} bins\n"
                    f"< {MIN_BINS}\nnot reported", transform=ax.get_xaxis_transform(),
                    ha="center", va="center", color=C_BAD, fontsize=10 * FONT_SCALE)
            pos += [j * (SEG_PARTS + 1) + p for p in range(SEG_PARTS)]
            lab += [""] * SEG_PARTS
            continue
        for p in range(SEG_PARTS):
            x = j * (SEG_PARTS + 1) + p
            pt, dr = SEG_PR[(oc, int(kk), p)]
            col = shade(CLUST_COLORS[j], 0.55 - 0.55 * p / max(SEG_PARTS - 1, 1))
            if len(dr):
                bp = ax.boxplot([dr], positions=[x], widths=0.72, patch_artist=True,
                                showfliers=False, zorder=3)
                bp["boxes"][0].set(facecolor=col, alpha=0.55, edgecolor=col, lw=1.4)
                for part in ("whiskers", "caps"):
                    for a_ in bp[part]:
                        a_.set(color=col, lw=1.3)
                bp["medians"][0].set(color=P["bg"], lw=0)
                ax.scatter(x + np.linspace(-0.16, 0.16, len(dr)), dr, s=3, lw=0,
                           color=col, alpha=0.35, zorder=4)
            ax.plot([x - 0.40, x + 0.40], [pt] * 2, color=P["fg"], lw=2.6, zorder=6)
            pos.append(x); lab.append(PART_NAMES[p])
    ax.set_xticks(pos); ax.set_xticklabels(lab, rotation=30, ha="right",
                                           fontsize=9.5 * FONT_SCALE)
    for j, kk in enumerate(ks):
        m = SEG_MATCH[(oc, int(kk))]
        ax.text(j * (SEG_PARTS + 1) + (SEG_PARTS - 1) / 2, 1.02,
                f"C{kk}" + (f"  (matched at {m:,} bins, {m / N_USED:.1f}×)" if m else ""),
                transform=ax.get_xaxis_transform(), ha="center", va="bottom",
                color=CLUST_COLORS[j], fontsize=10.5 * FONT_SCALE)
    ax.set_title(oc, fontsize=13 * FONT_SCALE, pad=24)
    ax.grid(axis="y", alpha=0.3)
for a in axes:      # the two panels are on very different scales, so both need the label
    a.set_ylabel(MEASURES["pr"]["label"], color=C_PR, fontsize=12.5 * FONT_SCALE)
for a, letter in zip(axes, "ab"):
    dw.label_panel(a, letter, color=P["fg"])
fig.suptitle("Participation ratio across a trial: early, middle and late, per cluster  "
             "(bar = point estimate, box = bootstrap over sessions)",
             fontsize=14.5 * FONT_SCALE)
print("Each cluster is matched to its OWN smallest third, so the three thirds of one cluster "
      "are\ncomparable and two clusters are not.\n")
st.save_fig(fig, FIGNAME); plt.show()

for oc in ["hit", "miss"]:
    for kk in np.unique(CLU[oc]["labels"]):
        if SEG_MATCH[(oc, int(kk))] is None:
            print(f"{oc:5s} C{kk}: not reported — its smallest third holds "
                  f"{min(SEG_N[(oc, int(kk), p)] for p in range(SEG_PARTS)):,} bins, floor is "
                  f"{MIN_BINS}")
            continue
        pts = [SEG_PR[(oc, int(kk), p)][0] for p in range(SEG_PARTS)]
        print(f"{oc:5s} C{kk}: " + "  ".join(f"{PART_NAMES[p]} {pts[p]:.1f}"
                                             for p in range(SEG_PARTS)), end="")
        a, b = SEG_PR[(oc, int(kk), 0)][1], SEG_PR[(oc, int(kk), SEG_PARTS - 1)][1]
        if len(a) and len(b):
            n = min(len(a), len(b))
            print(f"   | {PART_NAMES[-1]} − {PART_NAMES[0]} = {pts[-1] - pts[0]:+.1f}, "
                  f"bootstrap P(last > first) = {np.mean(b[:n] > a[:n]):.2f}")
        else:
            print()
print(f"\nPooling is ACROSS sessions and that is not optional: one session holds about "
      f"{int(np.median([SEG_N[k] for k in SEG_N if k[0] == 'hit']) / len(DB))} bins per "
      f"(cluster, third),\nagainst a floor of {MIN_BINS}. The box is a bootstrap over "
      f"sessions, so it carries session-to-session\nvariability; it is NOT a bin-level "
      f"bootstrap, which would be far too narrow. Clusters are largely\nduration clusters "
      f"(fig06/fig17), so 'early' in a long-trial cluster is more seconds than 'early' in a\n"
      f"short one — compare thirds WITHIN a cluster first.")

## 4 · Checks

In [ ]:

_OK = []


def check(name, ok, detail=""):
    _OK.append(bool(ok))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f"  — {detail}" if detail else ""))


# ---- conventions ----
check("the bin is 6 Hz, as in every other neural notebook here", abs(SRB - 6.0) < 1e-9,
      f"{BIN} frames at {SR:g} Hz = {SRB:g} Hz")
check("the warped window adds up", SPEC.n_cols == SPEC.pre_n + N_TRIAL_BINS + SPEC.post_n,
      f"{SPEC.pre_n} + {N_TRIAL_BINS} + {SPEC.post_n} = {SPEC.n_cols}")
check("window matrix and metadata are row-aligned",
      len(M_REAL) == len(SH) == len(M_SIM) == len(M_CLAMP) and len(SH) > 0,
      f"{len(SH)} trials")

# ---- the warp itself ----
_L, _n = 317, N_TRIAL_BINS                      # a trial long enough that columns span frames
_ramp = np.arange(_L, dtype=float)
_e = np.round(np.linspace(0, _L, _n + 1)).astype(int)
check("each trial column is the MEAN of the frames inside it (not one sampled frame)",
      np.allclose(warp_trial(_ramp, _n), [_ramp[a:b].mean() for a, b in zip(_e[:-1], _e[1:])]),
      f"exact on a {_L}-frame ramp warped to {_n} columns")
check("a short trial is interpolated up, and keeps its range",
      (lambda w: w.min() >= 0 and w.max() <= 9 and len(w) == _n)(warp_trial(np.arange(10.), _n)),
      "10 frames -> 50 columns, no extrapolation")
check("the window axis is one uniform grid of 1/n steps",
      np.allclose(np.diff(AX), 1.0 / N_TRIAL_BINS),
      f"step {np.diff(AX)[0]:.6f} = 1/{N_TRIAL_BINS}")
check("x = 0 is the first trial column and x = 1 is the first post-trial column",
      abs(AX[SPEC.pre_n]) < 1e-12 and abs(AX[SPEC.pre_n + N_TRIAL_BINS] - 1.0) < 1e-12,
      f"AX[{SPEC.pre_n}] = {AX[SPEC.pre_n]:.3f}, AX[{SPEC.pre_n + N_TRIAL_BINS}] = "
      f"{AX[SPEC.pre_n + N_TRIAL_BINS]:.3f}")
_S0 = bmi.session(MOUSE, int(REMAP[0]))
_t0 = trial_table(int(REMAP[0]))
_w, _k, _ = trial_windows(np.arange(_S0.nf, dtype=float), _t0.start.to_numpy(),
                          _t0.eff.to_numpy(), SPEC)
check("the flanks are real time and sit immediately either side of the trial",
      np.allclose(np.diff(_w[0][:SPEC.pre_n]), 1.0)
      and np.allclose(np.diff(_w[0][SPEC.pre_n + N_TRIAL_BINS:]), 1.0)
      and abs(_w[0][SPEC.pre_n + N_TRIAL_BINS] - round(_t0.eff.to_numpy()[_k[0]])) < 1e-6,
      "frame indices warped as a trace: flanks step by 1 frame, post starts at the trial end")

# ---- the reward clamp: the flat top is hardware, not smoothing ----
_cl = CLAMPED[(SH.outcome == "hit").to_numpy()]
# the last trial column spans trial_duration/N_TRIAL_BINS, i.e. 1.3 frames on a 2 s trial
# and 12 on a 20 s one, so a fixed ~2-frame clamp fills it on short trials and only part of it
# on long ones. What is being tested is WHERE the clamp sits, not how full one column is.
check("the clamp covers the end of a hit, not the middle of it",
      _cl[:, SPEC.trial_slice][:, -1].mean() > 0.6
      and _cl[:, SPEC.pre_n + N_TRIAL_BINS:].mean() > 0.5
      and _cl[:, :SPEC.pre_n].mean() < 0.02
      and _cl[:, SPEC.trial_slice][:, :N_TRIAL_BINS // 2].mean() < 0.02,
      f"{_cl[:, SPEC.trial_slice][:, -1].mean():.0%} of the last trial column, "
      f"{_cl[:, SPEC.pre_n + N_TRIAL_BINS:].mean():.0%} of the post flank, "
      f"{_cl[:, SPEC.trial_slice][:, :N_TRIAL_BINS // 2].mean():.1%} of the first half of "
      f"the trial, {_cl[:, :SPEC.pre_n].mean():.1%} of the pre flank")
# only the flanks are real frames; a warped trial column can blend clamped and unclamped
_post = slice(SPEC.pre_n + N_TRIAL_BINS, None)
_cv = M_REAL[(SH.outcome == "hit").to_numpy()][:, _post][_cl[:, _post]]
check("every clamped FRAME carries the same value — that is the flat top",
      float(np.nanstd(_cv)) / max(abs(float(np.nanmean(_cv))), 1e-9) < 0.01,
      f"{len(_cv):,} clamped post-flank frames, mean {np.nanmean(_cv):,.1f}, "
      f"sd {np.nanstd(_cv):,.3f}")
check("a miss is never clamped (it earns no reward)",
      CLAMPED[(SH.outcome == "miss").to_numpy()].mean() < 0.01,
      f"{CLAMPED[(SH.outcome == 'miss').to_numpy()].mean():.2%} of miss columns")
_Sc = bmi.session(MOUSE, int(STABLE[0]))
_tc = trial_table(int(STABLE[0]))
_raw, _kc, _ = trial_windows(_Sc.cur_raw, _tc.start.to_numpy(), _tc.eff.to_numpy(), SPEC)
_sm, _, _ = trial_windows(_Sc.cur, _tc.start.to_numpy(), _tc.eff.to_numpy(), SPEC)
check("section 2 is built from the UNSMOOTHED trace",
      np.allclose(M_REAL[(SH.i == int(STABLE[0])).to_numpy()][:len(_raw)], _raw, equal_nan=True)
      and not np.allclose(_raw, _sm, equal_nan=True),
      f"identical to the cur_raw windows; the smoothed ones differ by "
      f"{np.nanmean(np.abs(_raw - _sm)) / np.nanstd(_raw):.2f} sd")

# ---- the other-axis simulation ----
check("the simulation exists on remapping sessions and nowhere else",
      bool(SH[SH.i.isin(REMAP)].has_sim.any()) and not bool(SH[~SH.i.isin(REMAP)].has_sim.any()),
      f"{int(SH.has_sim.sum())} trials carry one, all on remapping sessions")
_cov = [float(np.mean(np.isfinite(np.asarray(bmi.simulate_other_axis(MOUSE, int(i)).D_other,
                                             float))[bmi.session(MOUSE, int(i)).in_exp]))
        for i in REMAP]
check("the simulation is defined on every in-session frame", min(_cov) > 0.999,
      f"worst session {min(_cov):.3%}")
_v = [bmi.simulate_other_axis(MOUSE, int(i)).val for i in REMAP]
check("the simulation still transfers to a block it was not fitted on",
      min(v["r_M"] for v in _v if v) > 0.85,
      f"held out on the washout, r(M) = {min(v['r_M'] for v in _v if v):.2f}–"
      f"{max(v['r_M'] for v in _v if v):.2f}")

# ---- the 5x rule ----
_rep_n = [int(ND[i]["pr_n"][list(ND[i]["pr_keys"]).index(f"{b}|session")])
          for i in sorted(DB) for b in PR_BLOCKS_OF[i]
          if np.isfinite(block_pr(i, b)) and f"{b}|session" in list(ND[i]["pr_keys"])]
check("no PR drawn in any panel comes from fewer than the required bins",
      min(_rep_n) >= MIN_BINS and MATCH["session"] >= MIN_BINS,
      f"floor = {PR_MIN_RATIO:g} x {N_USED} = {MIN_BINS} bins; smallest condition "
      f"{min(_rep_n)}, all matched down to {MATCH['session']}")
check("every reported PR of one kind used the SAME number of bins",
      all(MATCH[k] is None or MATCH[k] >= MIN_BINS for k in KINDS),
      "  ".join(f"{k}={MATCH[k]}" for k in KINDS))
_ib = SIZE[(SIZE.block == "intuitive") & (SIZE.kind == "all_trials")].n
check("the intuitive block's TRIAL-bins PR is refused, not fudged",
      not bool(MEAS[MEAS.block == "intuitive"].pr_trial_ok.any()),
      f"median {int(_ib.median())} trial bins = {_ib.median() / N_USED:.1f}x — which is why "
      f"panel A uses the block-level definition on both axes")
check("panel A compares spans of equal length, not just equal bin counts",
      set(MEAS.loc[MEAS.block == "new", "pr_src"]) == {"new_early"}
      and set(MEAS.loc[MEAS.block == "intuitive", "pr_src"]) == {"intuitive"},
      f"new axis drawn from its first {INT_MAX_MIN:g} min, like the intuitive block")
check("the ITI scenario is dropped entirely rather than reported badly",
      MATCH["iti"] is None,
      f"largest ITI bin count anywhere = {int(SIZE[SIZE.kind == 'iti'].n.max())}")
_pr1 = pr_matched(np.random.default_rng(1).normal(size=(6000, 40)), 3000, seed=7)[0]
_pr2 = pr_matched(np.random.default_rng(1).normal(size=(6000, 40)), 3000, seed=7)[0]
check("a matched PR is reproducible for a fixed seed", abs(_pr1 - _pr2) < 1e-9, f"{_pr1:.4f}")

# ---- the manifold measures ----
check("every correlation is a real correlation",
      bool((MEAS["corr"].dropna().abs() <= 1.0 + 1e-9).all()),
      f"range {MEAS['corr'].min():.3f} to {MEAS['corr'].max():.3f}")
check("all pairs are used, never a sample, at these trial counts",
      int(MEAS.n_trials.max()) <= PAIR_EXACT_MAX,
      f"largest condition = {int(MEAS.n_trials.max())} trials, exact up to {PAIR_EXACT_MAX}")
_tj = np.cumsum(np.ones((123, 3)), axis=0)
check("the trajectory warp keeps both endpoints of a trajectory",
      np.allclose(na.resample_trajectory(_tj, N_WARP)[[0, -1]], _tj[[0, -1]]),
      f"123 bins -> {N_WARP}, first and last unchanged")
check("trajectory length is positive everywhere it is reported",
      bool((MEAS["len"].dropna() > 0).all()), f"median {MEAS['len'].median():.1f}")
_vk = np.array([ND[i]["var_k"] for i in ND])
check(f"the {MANIFOLD_K}-PC manifold is the same size slice on every session",
      (_vk.max() - _vk.min()) < 0.15,
      f"it holds {_vk.min():.0%}–{_vk.max():.0%} of the variance; 80 % would need "
      f"{ELBOW80.min()}–{ELBOW80.max()} PCs, so this is a restricted view by choice")

# ---- block hygiene ----
check("the washout never enters any comparison",
      "washout" not in set(MEAS.block) and "washout" not in set(WINTAB.block),
      f"blocks used: {sorted(set(MEAS.block))}")
_int = TR[(TR.block == "intuitive") & TR.use]
check(f"the intuitive block is clipped to its first {INT_MAX_MIN:g} min",
      bool((_int.groupby("i").t_min.max() - _int.groupby("i").t_min.min()
            <= INT_MAX_MIN + 1e-6).all()),
      f"longest kept span {float((_int.groupby('i').t_min.max() - _int.groupby('i').t_min.min()).max()):.1f} min")
_bad = sum(not block_bin_span(DB[i], b)[s0:s1].all()
           for i, ws in WIN.items() for (b, s0, s1, _n) in ws)
check("no sliding window crosses a block boundary", _bad == 0,
      f"{sum(len(w) for w in WIN.values())} windows checked")
check("no bundled line mixes sessions from two phases, and none is used twice",
      (lambda a, b: len(a) == len(set(a)) and len(b) == len(set(b))
       and not (set(a) & set(b)))([i for g in bundles(STABLE, STABLE_BUNDLE, STABLE_N_BUNDLES)
                                   for i in g],
                                  [i for g in bundles(REMAP, REMAP_BUNDLE, REMAP_N_BUNDLES)
                                   for i in g]),
      f"fig14 {[len(g) for g in bundles(STABLE, STABLE_BUNDLE, STABLE_N_BUNDLES)]}, "
      f"fig16 {[len(g) for g in bundles(REMAP, REMAP_BUNDLE, REMAP_N_BUNDLES)]}")

# ---- clustering and abundance ----
check("every trial got a cluster label",
      all(len(CLU[o]["labels"]) == (SH.outcome == o).sum() for o in ["hit", "miss"]),
      "  ".join(f"{o}: {len(CLU[o]['labels'])}" for o in ["hit", "miss"]))
_ab = ca.cluster_composition(CLU["hit"]["sub"], "cluster", "i", normalize="index")
check("cluster abundances are fractions that sum to 1",
      bool(np.allclose(_ab.sum(axis=1).dropna(), 1.0)), f"{len(_ab)} sessions")
_bars = [CLU[oc]["sub"].pipe(lambda s, b=b, i=i: s[(s.block == b) & s.use & (s.i == i)])
         .cluster.value_counts(normalize=True).sum()
         for oc in ["hit", "miss"] for b in ["intuitive", "new"] for i in ABUND_I]
check("every stacked bar in fig18 sums to 1",
      all(abs(v - 1.0) < 1e-9 for v in _bars if np.isfinite(v) and v > 0),
      f"{sum(1 for v in _bars if np.isfinite(v) and v > 0)} bars")

# ---- the trial thirds ----
_db = DB[int(REMAP[0])]
_bi, _bt, _bp = trial_thirds(_db)
_j = int(_bt[0])
_jb, _jp = _bi[_bt == _j], _bp[_bt == _j]
check("the thirds of a trial are contiguous, disjoint and cover it",
      len(np.unique(_jp)) == SEG_PARTS and len(np.unique(_jb)) == _jb.size
      and np.all(np.diff(_jb) == 1) and np.all(np.diff(_jp) >= 0),
      f"trial {_j}: {_jb.size} bins over {SEG_PARTS} parts, none shared, none repeated")
check("the thirds are cut on the UNWARPED trial, so each is real time",
      abs(int((_bt == _j).sum()) - int(_db.eb[_j] - _db.sb[_j])) <= SEG_PARTS,
      f"{int((_bt == _j).sum())} labelled bins vs {int(_db.eb[_j] - _db.sb[_j])} trial bins")
for (_oc, _kk), _m in SEG_MATCH.items():
    _ns = [SEG_N[(_oc, _kk, p)] for p in range(SEG_PARTS)]
    if _m is not None:
        check(f"{_oc} C{_kk}: the thirds clear the floor and are matched to each other",
              _m >= MIN_BINS and all(n >= _m for n in _ns) and np.isfinite(
                  SEG_PR[(_oc, _kk, 0)][0]),
              f"matched at {_m:,} bins = {_m / N_USED:.1f}x, thirds hold "
              f"{min(_ns):,}–{max(_ns):,}")
    else:
        check(f"{_oc} C{_kk}: refused rather than reported below the floor",
              all(not np.isfinite(SEG_PR[(_oc, _kk, p)][0]) for p in range(SEG_PARTS)),
              f"smallest third {min(_ns):,} < {MIN_BINS}, and nothing is drawn for it")

print(f"\n{sum(_OK)}/{len(_OK)} checks pass")